# EE48009 — Assignment 1
## Channel Estimation and Pilot Overhead with Sionna

The notebook follows Steps 1–9 of the assignment. Run the cells from top to
bottom in a Colab GPU runtime. The ray-traced datasets, trained models and
link results are saved as `.pt` files.


In [ ]:
# Sionna 2.2 uses PyTorch. Install all pinned versions in one cell.
%pip install -q "sionna==2.2.0" "numpy>=2.2.6,<2.3"


In [ ]:
# If Colab asks for a runtime restart after installation, restart once and
# continue from this cell.
import sys
print("Python:", sys.version.split()[0])


## Step 1 — Scenario selection

I use Sionna's built-in San Francisco scene at 3.5 GHz. The scene contains
streets, buildings and corners, so direct, reflected and diffracted paths
can occur. Different excess delays make the channel frequency-selective.
UE motion changes the path phases and makes the channel time-selective.

A 30 kHz subcarrier spacing and one 14-symbol slot are used below. I do not
assume beforehand that all points are LoS or NLoS: both groups are checked
after the BS and UE grid are placed.


In [ ]:
import sionna
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, \
                      PathSolver, RadioMapSolver, subcarrier_frequencies

scene = load_scene(sionna.rt.scene.san_francisco)
scene.frequency = 3.5e9

In [ ]:
#Obtain the bounding box for the scene
bbox = scene.mi_scene.bbox()

#Minimum and maximum coordinates
p_min = np.array(bbox.min, dtype=float)
p_max = np.array(bbox.max, dtype=float)

#Calculate the scene center and dimensions
center = (p_min + p_max) / 2
size = p_max - p_min

print("Scene center:", center)
print("Scene dimensions:", size)

#Automatically position the camera
camera_position = center + np.array([0.0, -1 * max(size[0], size[1]), 1 * max(size[0], size[1])])

auto_cam = Camera(
    position=camera_position.tolist(),
    look_at=center.tolist()
)

scene.render(
    camera=auto_cam,
    resolution=[650, 550],
    num_samples=128
);


## Step 2 — Base Station and UE Configuration

### 2.1 Antenna Array Configuration

In [ ]:

from sionna.rt import PlanarArray

#Creating a single transmit antenna of UE
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

#BS: M x M receive antenna array
M = 1  #M will take values 1,2 and 4.

scene.rx_array = PlanarArray(
    num_rows=M,
    num_cols=M,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="V"
)

print(f"UE antennas: 1")
print(f"BS array: {M} x {M}")
print(f"BS antenna elements: {M**2}")


### 2.2 Base Station Placement

The roof position selected in Blender is `[62.16, 126.9, 28.3481]` m.
With a 1.5 m rooftop mast, the antenna position is
`[62.16, 126.9, 29.8481]` m. The array is aimed at the centre of the UE
region after the UE heights are assigned.


In [ ]:
import mitsuba as mi
from sionna.rt import Receiver

def top_surface_z(x, y):
    # Highest scene surface at this (x,y). This avoids putting a BS inside
    # a roof or inside the terrain.
    z_start = float(np.asarray(scene.mi_scene.bbox().max)[2]) + 20.0
    ray = mi.Ray3f(
        o=mi.Point3f(float(x), float(y), z_start),
        d=mi.Vector3f(0.0, 0.0, -1.0),
    )
    hit = scene.mi_scene.ray_intersect(ray)
    if not bool(np.asarray(hit.is_valid()).item()):
        raise RuntimeError("No scene surface was found below the BS point.")
    return float(np.asarray(hit.p.z).item())

# Site-A roof position and mast height
BS_ROOF_POSITION = np.array([62.16, 126.9, 28.3481])
BS_MAST_HEIGHT_M = 1.5
BS_A_POSITION = BS_ROOF_POSITION + np.array([0.0, 0.0, BS_MAST_HEIGHT_M])
BS_A_XY = BS_A_POSITION[:2]
bs_position = BS_A_POSITION.tolist()

scene.add(Receiver(
    name="bs",
    position=bs_position,
    color=(0.1, 0.8, 0.1),
    display_radius=2.0,
))
bs = scene.get("bs")
print("Selected roof point:", BS_ROOF_POSITION)
print("Scene roof surface z:", round(top_surface_z(*BS_A_XY), 4))
print("Rooftop mast height:", BS_MAST_HEIGHT_M, "m")
print("BS antenna position:", bs.position)


In [ ]:

#This cell visualises the BS.

scene.render(
    camera=auto_cam,
    show_devices=True,
    resolution=[1000, 750],
    num_samples=128
);


## Step 3.1 — UE coverage region

In [ ]:
import numpy as np

# UE grid parameters
center_x = 20.0
center_y = 283.0
plane_center_z = 30.0

region_size = 300.0
grid_spacing = 10.0

rotation_deg = 10.0
rotation_rad = np.deg2rad(rotation_deg)

ue_region_center = [center_x, center_y]
blender_plane_center = [center_x, center_y, plane_center_z]

print("Blender plane center:", blender_plane_center)
print("Dimensions:", region_size, "x", region_size, "m")
print("Rotation:", rotation_deg, "degrees")



## Step 3.2 — Terrain-following UE surface

In [ ]:
import numpy as np
from sionna.rt import transform_mesh

#Copy the San Francisco terrain
measurement_surface = scene.objects["Terrain"].clone(
    as_mesh=True
)

#Raise it by 1.5 meters to represent UE height
transform_mesh(
    measurement_surface,
    translation=np.array([0, 0, 1.5])
)

print("UE measurement surface created.")


## Step 3.3 — Candidate UE grid

In [ ]:

# Step 3.3 — Generate rotated 2D UE grid

points_per_side = int(round(region_size / grid_spacing)) + 1
local_values = np.linspace(
    -region_size / 2,
    region_size / 2,
    points_per_side,
)

# Local coordinates before rotation
U, V = np.meshgrid(local_values, local_values)

# Rotate into world coordinates
X = (
    center_x
    + U * np.cos(rotation_rad)
    - V * np.sin(rotation_rad)
)

Y = (
    center_y
    + U * np.sin(rotation_rad)
    + V * np.cos(rotation_rad)
)

# Global XY coordinates used by Sionna
ue_xy = np.column_stack((X.ravel(), Y.ravel()))

# Local coordinates used for train/val/test splitting
ue_local = np.column_stack((U.ravel(), V.ravel()))

expected_candidates = points_per_side ** 2
assert len(ue_xy) == expected_candidates == 961
print("Candidate UE locations:", len(ue_xy))
print("First five positions:")
print(ue_xy[:5])



The resulting candidate grid is shown below.

## Step 3.4 — Assign UE heights using the terrain

In [ ]:
import mitsuba as mi

terrain_scene = mi.load_dict({
    "type": "scene",
    "terrain": measurement_surface
})

ue_positions = []
terrain_grid_indices = []
missing_terrain = []

ray_start_z = 1000.0

for grid_idx, (x, y) in enumerate(ue_xy):

    ray = mi.Ray3f(
        o=mi.Point3f(float(x), float(y), ray_start_z),
        d=mi.Vector3f(0.0, 0.0, -1.0)
    )

    hit = terrain_scene.ray_intersect(ray)

    if bool(np.asarray(hit.is_valid()).item()):

        z = float(np.asarray(hit.p.z).item())

        ue_positions.append([float(x), float(y), z])
        terrain_grid_indices.append(grid_idx)

    else:
        missing_terrain.append([float(x), float(y)])

ue_positions = np.array(ue_positions).reshape(-1, 3)
terrain_grid_indices = np.array(
    terrain_grid_indices, dtype=int
)

print("Candidates:", len(ue_xy))
print("Terrain intersections:", len(ue_positions))
print("Missing terrain:", len(missing_terrain))



## Step 3.5 — Filter indoor UE positions

In [ ]:

# Step 3.5 — Filter likely indoor UE locations

outdoor_positions = []
blocked_positions = []

outdoor_grid_indices = []

for idx, pos in enumerate(ue_positions):

    x, y, z = pos

    ray = mi.Ray3f(
        o=mi.Point3f(
            float(x),
            float(y),
            float(z + 0.05)
        ),
        d=mi.Vector3f(0.0, 0.0, 1.0)
    )

    hit = scene.mi_scene.ray_test(ray)
    is_blocked = bool(np.asarray(hit).item())

    if is_blocked:
        blocked_positions.append(pos)
    else:
        outdoor_positions.append(pos)

        # Keep the corresponding rotated-grid index
        outdoor_grid_indices.append(
            terrain_grid_indices[idx]
        )

outdoor_positions = np.array(
    outdoor_positions
).reshape(-1, 3)

blocked_positions = np.array(
    blocked_positions
).reshape(-1, 3)

outdoor_grid_indices = np.array(
    outdoor_grid_indices, dtype=int
)

# Matching local coordinates for surviving UE points
outdoor_local = ue_local[outdoor_grid_indices]

print("Total positions:", len(ue_positions))
print("Outdoor candidates:", len(outdoor_positions))
print("Excluded:", len(blocked_positions))



In [ ]:
# Set the Site-A BS position and orientation
bs = scene.get("bs")
bs.position = BS_A_POSITION.tolist()

aim_point = np.array([
    center_x,
    center_y,
    float(np.median(outdoor_positions[:, 2])),
])
bs.look_at(aim_point.tolist())

print("Final BS position:", bs.position)
print("Aim point:", aim_point)
print("BS orientation:", bs.orientation)


## Step 3.6 — UE velocities

In [ ]:
# Same random direction for all speeds at a given location

rng = np.random.default_rng(42)

speeds_kmh = [3, 30, 120]
ue_dataset = []

for location_id, position in enumerate(outdoor_positions):

    # One random direction per UE location
    theta = rng.uniform(0, 2 * np.pi)

    for speed_kmh in speeds_kmh:

        speed_ms = speed_kmh / 3.6

        velocity = speed_ms * np.array([
            np.cos(theta),
            np.sin(theta),
            0.0
        ])

        ue_dataset.append({
            "location_id": location_id,
            "position": position.copy(),
            "speed_kmh": speed_kmh,
            "velocity": velocity,
            "direction_rad": theta
        })

print("Total channel cases:", len(ue_dataset))

for sample in ue_dataset[:3]:
    print(sample)


## Step 3.8 — Spatially disjoint dataset split

In [ ]:

# Step 3.8 — Spatial split in rotated local coordinates

location_splits = {}

for location_id, local_position in enumerate(outdoor_local):

    u, v = local_position

    if v <= -110.0:
        split = "test"

    elif -90.0 <= v <= 90.0:
        split = "train"

    elif v >= 110.0:
        split = "validation"

    else:
        split = "buffer"

    location_splits[location_id] = split

# Assign all three speeds at a location to the same split
for sample in ue_dataset:
    sample["split"] = location_splits[sample["location_id"]]

train_data = [
    s for s in ue_dataset if s["split"] == "train"
]

val_data = [
    s for s in ue_dataset if s["split"] == "validation"
]

test_data = [
    s for s in ue_dataset if s["split"] == "test"
]

from collections import Counter

counts = Counter(location_splits.values())

print("Location counts:", counts)
print("Training channel cases:", len(train_data))
print("Validation channel cases:", len(val_data))
print("Test channel cases:", len(test_data))

assert len(location_splits) == len(outdoor_positions)
assert sum(counts.values()) == len(outdoor_positions)



In [ ]:

# Step 3.9 — Visualize rotated spatial dataset split

import matplotlib.pyplot as plt

colors = {
    "train": "royalblue",
    "validation": "seagreen",
    "test": "darkorange",
    "buffer": "lightgray"
}

plt.figure(figsize=(9, 8))

for split, color in colors.items():

    positions = np.array([
        position
        for location_id, position in enumerate(outdoor_positions)
        if location_splits[location_id] == split
    ]).reshape(-1, 3)

    if len(positions) > 0:
        plt.scatter(
            positions[:, 0],
            positions[:, 1],
            color=color,
            s=24,
            label=f"{split.capitalize()} ({len(positions)})"
        )

# Draw rotated square boundary
half_size = region_size / 2.0
corners_local = np.array([
    [-half_size, -half_size],
    [ half_size, -half_size],
    [ half_size,  half_size],
    [-half_size,  half_size],
    [-half_size, -half_size]
])

c = np.cos(rotation_rad)
s = np.sin(rotation_rad)

corners_global = np.column_stack((
    center_x + corners_local[:, 0]*c - corners_local[:, 1]*s,
    center_y + corners_local[:, 0]*s + corners_local[:, 1]*c
))

plt.plot(
    corners_global[:, 0],
    corners_global[:, 1],
    color="black",
    linewidth=1.5,
    linestyle="--",
    label="UE region boundary"
)

bs_xyz = bs.position.numpy().reshape(-1)

plt.scatter(
    bs_xyz[0], bs_xyz[1],
    color="black",
    marker="^",
    s=150,
    label="BS A (selected rooftop)"
)

plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Rotated Spatial Train / Validation / Test Split")

plt.axis("equal")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# Visualize the UE grid and BS in the 3-D scene
splits_vis = np.array([location_splits[i] for i in range(len(outdoor_positions))])
split_colors = {
    "train": (0.25, 0.41, 0.88),
    "validation": (0.18, 0.55, 0.34),
    "test": (1.0, 0.55, 0.0),
    "buffer": (0.6, 0.6, 0.6),
}

viz_names = []
for i, (p, s) in enumerate(zip(outdoor_positions, splits_vis)):
    name = f"viz_{i}"
    scene.add(Transmitter(
        name=name,
        position=[float(x) for x in p],
        color=split_colors[s],
        display_radius=2.0,
    ))
    viz_names.append(name)

grid_center = outdoor_positions.mean(0)
bs_xyz = scene.get("bs").position.numpy().reshape(-1)

try:
    top_cam = Camera(
        position=[float(grid_center[0]), float(grid_center[1]) - 1.0, 600.0],
        look_at=[float(grid_center[0]), float(grid_center[1]), 0.0],
    )
    scene.render(
        camera=top_cam, show_devices=True,
        resolution=(1000, 750), num_samples=128,
    )

    horizontal = (bs_xyz - grid_center) * np.array([1, 1, 0])
    horizontal /= np.linalg.norm(horizontal) + 1e-9
    side_cam = Camera(
        position=(bs_xyz + horizontal * 120 + np.array([0, 0, 60])).tolist(),
        look_at=grid_center.tolist(),
    )
    scene.render(
        camera=side_cam, show_devices=True,
        resolution=(1000, 750), num_samples=128,
    )
finally:
    for name in viz_names:
        if name in scene.transmitters:
            scene.remove(name)


## Step 3.10 — Radio map

The BS is temporarily used as a transmitter to plot reciprocal path gain
over the UE surface. The map shows building shadowing and helps interpret
the LoS and NLoS locations.

In [ ]:
def to_np(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    return np.asarray(value)

saved_tx_array = scene.tx_array
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="tr38901", polarization="V",
)
bs_tx = Transmitter(
    name="bs_tx",
    position=scene.get("bs").position.numpy().reshape(-1).tolist(),
)
scene.add(bs_tx)
bs_tx.look_at(aim_point.tolist())

try:
    rm_solver = RadioMapSolver()
    radio_map = rm_solver(
        scene,
        measurement_surface=measurement_surface,
        samples_per_tx=2_000_000,
        max_depth=5,
        los=True,
        specular_reflection=True,
        diffuse_reflection=False,
        refraction=False,
        diffraction=True,
        seed=42,
    )
    print("Radio map computed")

    path_gain = to_np(radio_map.path_gain)
    gain_db = 10 * np.log10(path_gain[path_gain > 0])
    vmin = float(np.percentile(gain_db, 2))
    vmax = float(gain_db.max())
    print(f"Path-gain range: {vmin:.0f} to {vmax:.0f} dB")

    top_cam_rm = Camera(
        position=[float(center[0]), float(center[1]) - 1.0, 1100.0],
        look_at=[float(center[0]), float(center[1]), 0.0],
    )
    for camera in (top_cam_rm, auto_cam):
        scene.render(
            camera=camera,
            radio_map=radio_map,
            rm_metric="path_gain",
            rm_db_scale=True,
            rm_vmin=vmin,
            rm_vmax=vmax,
            show_devices=True,
            resolution=(1000, 750),
            num_samples=128,
        )
    plt.show()
finally:
    if "bs_tx" in scene.transmitters:
        scene.remove("bs_tx")
    scene.tx_array = saved_tx_array


In [ ]:
# Check the BS orientation
bs_xyz = scene.get("bs").position.numpy().reshape(-1)
target_xyz = np.asarray(aim_point)
delta = target_xyz - bs_xyz
horizontal_distance = np.linalg.norm(delta[:2])
downtilt_deg = np.degrees(np.arctan2(-delta[2], horizontal_distance))

print("BS position:", bs_xyz)
print("Target:", target_xyz)
print("Horizontal distance:", round(horizontal_distance, 1), "m")
print("Downtilt:", round(downtilt_deg, 2), "degrees")
print("Sionna orientation:", scene.get("bs").orientation)

if downtilt_deg <= 0:
    raise RuntimeError("The BS is not above the UE target. Check its height.")


In [ ]:
# Save the spatial split. Downloading is optional in Colab.
import pandas as pd

rows = []
for location_id, position in enumerate(outdoor_positions):
    rows.append({
        "location_id": location_id,
        "x": float(position[0]),
        "y": float(position[1]),
        "z": float(position[2]),
        "split": location_splits[location_id],
    })

split_table = pd.DataFrame(rows)
csv_filename = "ue_spatial_split.csv"
split_table.to_csv(csv_filename, index=False)
print("Saved", csv_filename)
print(split_table["split"].value_counts())

try:
    from google.colab import files
    # Uncomment only when a local copy is needed.
    # files.download(csv_filename)
except ImportError:
    pass


## Step 4 — Ray tracing and OFDM channels

The PathSolver uses LoS, specular reflection and diffraction with
max_depth=4. Refraction and diffuse reflection are disabled. The CIR is
sampled at the OFDM-symbol rate for 14 time steps and converted to 288
subcarriers in small UE chunks.

No-path positions are kept in the raw dataset with a valid_path flag. They
are shown and counted, but excluded from normalized-channel NMSE because a
zero channel cannot be normalized. LoS labels come from a direct geometric
visibility ray rather than a delay tolerance.


In [ ]:
import math
import os
import random
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 41
random.seed(SEED)
np.random.seed(SEED)

FC = 3.5e9
C0 = 299_792_458.0
SCS = 30e3
N_PRB = 24
NSC = 12 * N_PRB
NSYM = 14
T_SLOT = 0.5e-3
T_SYM = T_SLOT / NSYM
SPEEDS = [3, 30, 120]
SNR_GRID = list(range(-5, 26, 5))
FREQS = (np.arange(NSC) - NSC / 2) * SCS
FD_NOM = {v: v / 3.6 * FC / C0 for v in SPEEDS}

def to_numpy(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def complex_noise(shape, variance, rng):
    z = rng.standard_normal(shape) + 1j * rng.standard_normal(shape)
    return (z * np.sqrt(variance / 2)).astype(np.complex64)

def path_stats(power, delay, doppler, dynamic_range_db=30.0):
    pmax = power.max(axis=1, keepdims=True)
    keep = (power > 0) & (
        power >= pmax * 10 ** (-dynamic_range_db / 10)
    )
    w = np.where(keep, power, 0.0)
    total = np.maximum(w.sum(axis=1), 1e-30)

    def moment(x, order):
        return (w * np.where(keep, x, 0.0) ** order).sum(axis=1) / total

    mean_tau = moment(delay, 1)
    mean_tau2 = moment(delay, 2)
    return {
        "tau_rms": np.sqrt(np.maximum(mean_tau2 - mean_tau**2, 0.0)),
        "tau_max": np.where(keep, delay, 0.0).max(axis=1),
        "fd_rms": np.sqrt(moment(doppler, 2)),
        "fd_max": np.where(keep, np.abs(doppler), 0.0).max(axis=1),
    }

def subarray(H, M):
    # Central nested sub-arrays of the traced 4x4 array.
    indices = {
        1: [5],
        2: [5, 6, 9, 10],
        4: list(range(16)),
    }[M]
    out = H[:, indices]
    power = np.mean(np.abs(out) ** 2, axis=(1, 2, 3), keepdims=True)
    return (out / np.sqrt(np.maximum(power, 1e-30))).astype(np.complex64)

print("Nominal maximum Doppler [Hz]:",
      {v: round(fd, 1) for v, fd in FD_NOM.items()})


In [ ]:
import torch
from sionna.phy import config as sionna_config
from sionna.phy.channel import cir_to_ofdm_channel
from sionna.rt import PathSolver, PlanarArray, Transmitter

device = "cuda:0" if torch.cuda.is_available() else "cpu"
sionna_config.device = device
torch.manual_seed(SEED)
sionna_config.seed = SEED
print("Device:", device)

path_solver = PathSolver()
path_options = {
    "max_depth": 4,
    "los": True,
    "specular_reflection": True,
    "diffuse_reflection": False,
    "refraction": False,
    "diffraction": True,
    "samples_per_src": 500_000,
}
ofdm_frequencies = torch.tensor(
    FREQS, dtype=torch.float32, device=device
)

def direct_los(scene, ue_positions, bs_position):
    labels = []
    bs_position = np.asarray(bs_position, dtype=float)
    for position in np.asarray(ue_positions, dtype=float):
        direction = bs_position - position
        distance = np.linalg.norm(direction)
        direction = direction / distance
        origin = position + 0.05 * direction
        ray = mi.Ray3f(
            o=mi.Point3f(*[float(x) for x in origin]),
            d=mi.Vector3f(*[float(x) for x in direction]),
        )
        ray.maxt = float(max(distance - 0.10, 0.01))
        blocked = bool(np.asarray(scene.mi_scene.ray_test(ray)).item())
        labels.append(not blocked)
    return np.asarray(labels, dtype=bool)

def trace_paths(scene, seed):
    return path_solver(scene, seed=int(seed), **path_options)

def generate_channels(
    scene,
    positions,
    directions,
    speed_kmh,
    los_labels,
    site_seed,
    chunk_size=24,
):
    channels = []
    valid_rows = []
    gains = []
    statistics = {k: [] for k in ("tau_rms", "tau_max", "fd_rms", "fd_max")}

    for start in range(0, len(positions), chunk_size):
        stop = min(start + chunk_size, len(positions))
        ids = np.arange(start, stop)
        names = []

        try:
            for i in ids:
                name = f"ue_{i}"
                names.append(name)
                scene.add(Transmitter(
                    name=name,
                    position=[float(x) for x in positions[i]],
                ))
                speed_ms = speed_kmh / 3.6
                velocity = speed_ms * np.array([
                    np.cos(directions[i]),
                    np.sin(directions[i]),
                    0.0,
                ])
                scene.get(name).velocity = [float(x) for x in velocity]

            # The seed depends on the site and chunk, not on the speed.
            paths = trace_paths(scene, site_seed + start)
            a, tau = paths.cir(
                sampling_frequency=1 / T_SYM,
                num_time_steps=NSYM,
                normalize_delays=False,
                out_type="torch",
            )
            a = torch.as_tensor(a, device=device)
            tau = torch.as_tensor(tau, device=device)

            if tau.ndim != 3:
                raise RuntimeError("Unexpected CIR delay shape.")

            count = len(ids)
            if tau.shape[-1] == 0:
                channels.append(np.zeros(
                    (count, scene.rx_array.num_ant, NSYM, NSC),
                    dtype=np.complex64,
                ))
                valid_rows.append(np.zeros(count, dtype=bool))
                gains.append(np.full(count, -300.0))
                for key in statistics:
                    statistics[key].append(np.zeros(count))
                continue

            # a: [rx, rx_ant, tx, tx_ant, path, time]
            # tau: [rx, tx, path]
            path_power = (a.abs() ** 2).mean(dim=(1, 5))[0, :, 0, :]
            tau0 = tau[0]
            valid_path = (tau0 >= 0) & (path_power > 0)
            first_delay = torch.where(
                valid_path,
                tau0,
                torch.full_like(tau0, 1e9),
            ).min(dim=-1).values
            excess_delay = torch.where(
                valid_path,
                tau0 - first_delay[:, None],
                torch.zeros_like(tau0),
            )

            phase_step = (
                a[..., 1:] * a[..., :-1].conj()
            ).sum(dim=(1, 5))[0, :, 0, :]
            doppler = torch.angle(phase_step) / (2 * np.pi * T_SYM)

            a_batch = a.permute(2, 0, 1, 3, 4, 5).unsqueeze(3)
            h = cir_to_ofdm_channel(
                ofdm_frequencies,
                a_batch,
                excess_delay[:, None, None, :],
                normalize=False,
            )
            h = h[:, 0, :, 0, 0]
            power = (h.abs() ** 2).mean(dim=(1, 2, 3))
            ok = valid_path.any(dim=-1) & (power > 1e-20)
            scale = torch.where(ok, torch.sqrt(power), torch.ones_like(power))
            h = h / scale[:, None, None, None]

            stats = path_stats(
                to_numpy(path_power),
                to_numpy(excess_delay),
                to_numpy(doppler),
            )
            for key in statistics:
                statistics[key].append(stats[key])

            channels.append(to_numpy(h).astype(np.complex64))
            valid_rows.append(to_numpy(ok).astype(bool))
            gains.append(10 * np.log10(to_numpy(power) + 1e-30))
        finally:
            for name in names:
                if name in scene.transmitters:
                    scene.remove(name)

    result = {
        "H": np.concatenate(channels),
        "valid_path": np.concatenate(valid_rows),
        "gain_db": np.concatenate(gains),
        "los": np.asarray(los_labels, dtype=bool),
    }
    for key, values in statistics.items():
        result[key] = np.concatenate(values)
    return result

def generate_dataset(
    scene,
    positions,
    splits,
    location_directions,
    los_labels,
    speeds=SPEEDS,
    site_seed=1000,
    chunk_size=24,
):
    columns = {}
    started = time.time()

    selected = np.where(splits != "buffer")[0]
    for speed in speeds:
        result = generate_channels(
            scene,
            positions[selected],
            location_directions[selected],
            speed,
            los_labels[selected],
            site_seed,
            chunk_size,
        )
        count = len(selected)
        metadata = {
            "speed": np.full(count, speed),
            "loc": selected.copy(),
            "split": splits[selected].copy(),
            "dir_id": np.zeros(count, dtype=int),
            "pos": positions[selected].copy(),
        }
        for key, value in {**result, **metadata}.items():
            columns.setdefault(key, []).append(value)

        n_valid = int(result["valid_path"].sum())
        print(
            f"{speed:>3} km/h: {n_valid}/{count} locations have paths "
            f"[{time.time() - started:.0f} s]"
        )

    return {key: np.concatenate(value) for key, value in columns.items()}


In [ ]:
# Trace the 4x4 array once. M=2 uses its central 2x2 block and
# M=1 uses one central element.
M_TRACE = 4
scene.rx_array = PlanarArray(
    num_rows=M_TRACE,
    num_cols=M_TRACE,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="V",
)

bs = scene.get("bs")
bs_pos_A = bs.position.numpy().reshape(-1)
splits_arr = np.array([
    location_splits[i] for i in range(len(outdoor_positions))
])
loc_dirs = np.array([
    ue_dataset[3 * i]["direction_rad"]
    for i in range(len(outdoor_positions))
])

los_A = direct_los(scene, outdoor_positions, bs_pos_A)
used = splits_arr != "buffer"
print("Geometric LoS fraction (non-buffer):", round(los_A[used].mean(), 3))
print("LoS locations:", int(los_A[used].sum()))
print("NLoS locations:", int((~los_A[used]).sum()))

if los_A[used].all() or (~los_A[used]).all():
    raise RuntimeError(
        "The selected geometry does not contain both LoS and NLoS points."
    )


In [ ]:
CACHE_FORMAT = 1
SITE_A_FILE = "siteA_channels_4x4.pt"
site_a_meta = {
    "format": CACHE_FORMAT,
    "bs_position": bs_pos_A.tolist(),
    "frequency": FC,
    "subcarrier_spacing": SCS,
    "num_subcarriers": NSC,
    "num_symbols": NSYM,
    "solver": path_options,
    "ue_center": [center_x, center_y],
    "region_size": region_size,
    "grid_spacing": grid_spacing,
    "rotation_deg": rotation_deg,
    "candidate_count": len(ue_xy),
    "outdoor_count": len(outdoor_positions),
    "outdoor_positions": np.round(outdoor_positions, 4).tolist(),
    "location_splits": splits_arr.tolist(),
}

def cache_matches(dataset, expected):
    meta = dataset.get("metadata", {})
    if meta.get("format") != expected["format"]:
        return False
    if not np.allclose(
        meta.get("bs_position", [np.nan] * 3),
        expected["bs_position"],
        atol=1e-4,
    ):
        return False
    if not np.allclose(
        meta.get("ue_center", [np.nan] * 2),
        expected["ue_center"],
        atol=1e-4,
    ):
        return False
    saved_positions = np.asarray(meta.get("outdoor_positions", []))
    expected_positions = np.asarray(expected["outdoor_positions"])
    if (
        saved_positions.shape != expected_positions.shape
        or not np.allclose(saved_positions, expected_positions, atol=1e-4)
    ):
        return False
    return (
        meta.get("frequency") == expected["frequency"]
        and meta.get("subcarrier_spacing") == expected["subcarrier_spacing"]
        and meta.get("num_subcarriers") == expected["num_subcarriers"]
        and meta.get("num_symbols") == expected["num_symbols"]
        and meta.get("solver") == expected["solver"]
        and meta.get("region_size") == expected["region_size"]
        and meta.get("grid_spacing") == expected["grid_spacing"]
        and meta.get("rotation_deg") == expected["rotation_deg"]
        and meta.get("candidate_count") == expected["candidate_count"]
        and meta.get("outdoor_count") == expected["outdoor_count"]
        and meta.get("location_splits") == expected["location_splits"]
    )

if os.path.exists(SITE_A_FILE):
    ds = torch.load(SITE_A_FILE, weights_only=False)
    if not cache_matches(ds, site_a_meta):
        raise RuntimeError(
            f"{SITE_A_FILE} belongs to another geometry. Rename or delete it."
        )
    print("Loaded", SITE_A_FILE)
else:
    ds = generate_dataset(
        scene,
        outdoor_positions,
        splits_arr,
        loc_dirs,
        los_A,
        site_seed=1000,
    )
    ds["metadata"] = site_a_meta
    torch.save(ds, SITE_A_FILE)
    print("Saved", SITE_A_FILE)

base_sets = []
for speed in SPEEDS:
    mask = (ds["speed"] == speed) & ds["valid_path"]
    base_sets.append(set(ds["loc"][mask].tolist()))
if not all(s == base_sets[0] for s in base_sets[1:]):
    raise RuntimeError("The valid location set changed between speeds.")

raw_locations = int((splits_arr != "buffer").sum())
valid_locations = len(base_sets[0])
path_fraction = valid_locations / raw_locations
print(f"Valid locations: {valid_locations}/{raw_locations} ({path_fraction:.1%})")
print("No-path locations:", raw_locations - valid_locations)

valid_base = (
    (ds["speed"] == SPEEDS[0])
    & ds["valid_path"]
    & (ds["dir_id"] == 0)
)
if path_fraction < 0.80:
    print("Warning: fewer than 80% of locations have a path.")
if ds["los"][valid_base].all() or (~ds["los"][valid_base]).all():
    raise RuntimeError("Valid channels do not contain both LoS and NLoS.")

no_path_locs = sorted(
    set(np.where(splits_arr != "buffer")[0]) - base_sets[0]
)
if no_path_locs:
    p = outdoor_positions[no_path_locs]
    plt.figure(figsize=(7, 6))
    plt.scatter(outdoor_positions[:, 0], outdoor_positions[:, 1],
                s=12, color="lightgray", label="all outdoor points")
    plt.scatter(p[:, 0], p[:, 1], s=30, color="red",
                label=f"no path ({len(p)})")
    plt.scatter(bs_pos_A[0], bs_pos_A[1], marker="^", s=120,
                color="black", label="BS A")
    plt.axis("equal")
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.legend()
    plt.title("No-path locations")
    plt.show()


In [ ]:
# Plot a subset of propagation paths.
from sionna.rt import Camera

picture_positions = outdoor_positions[::8]
picture_names = []
try:
    for i, position in enumerate(picture_positions):
        name = f"path_picture_{i}"
        picture_names.append(name)
        scene.add(Transmitter(
            name=name,
            position=[float(x) for x in position],
        ))

    picture_paths = path_solver(
        scene,
        max_depth=3,
        los=True,
        specular_reflection=True,
        diffuse_reflection=False,
        refraction=False,
        diffraction=True,
        synthetic_array=False,
        samples_per_src=100_000,
        seed=SEED,
    )
    grid_center = outdoor_positions.mean(axis=0)
    horizontal = (bs_pos_A - grid_center) * np.array([1, 1, 0])
    horizontal /= np.linalg.norm(horizontal) + 1e-9
    path_camera = Camera(
        position=(bs_pos_A + 120 * horizontal + [0, 0, 60]).tolist(),
        look_at=grid_center.tolist(),
    )
    scene.render(
        camera=path_camera,
        paths=picture_paths,
        show_devices=True,
        resolution=(1000, 700),
        num_samples=128,
    )
    plt.show()
finally:
    for name in picture_names:
        if name in scene.transmitters:
            scene.remove(name)


In [ ]:
def draw_cdf(ax, values, label):
    values = np.sort(values)
    prob = np.arange(1, len(values) + 1) / len(values)
    ax.plot(values, prob, label=f"{label} (n={len(values)})")

def plot_channel_statistics(dataset):
    selected = (
        (dataset["speed"] == SPEEDS[0])
        & (dataset["dir_id"] == 0)
        & dataset["valid_path"]
    )
    tau_rms = dataset["tau_rms"][selected]
    tau_max = dataset["tau_max"][selected]
    los = dataset["los"][selected]
    coherence_bw_sc = 1 / (5 * np.maximum(tau_rms, 1e-12)) / SCS

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    for group, label in ((los, "LoS"), (~los, "NLoS")):
        draw_cdf(axes[0], tau_rms[group] * 1e9, label)
        draw_cdf(axes[1], coherence_bw_sc[group], label)
        draw_cdf(axes[2], tau_max[group] * 1e9, label)

    axes[0].set_xlabel("RMS delay spread [ns]")
    axes[1].set_xlabel("Coherence bandwidth [subcarriers]")
    axes[1].set_xscale("log")
    axes[2].set_xlabel("Maximum excess delay [ns]")
    for ax in axes:
        ax.set_ylabel("CDF")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

    print("LoS fraction:", round(los.mean(), 3))
    for group, label in ((los, "LoS"), (~los, "NLoS")):
        print(
            f"{label}: median delay spread "
            f"{np.median(tau_rms[group]) * 1e9:.1f} ns, "
            f"90% max delay {np.percentile(tau_max[group], 90) * 1e9:.1f} ns"
        )

    print("\nTime selectivity:")
    for speed in SPEEDS:
        fd = FD_NOM[speed]
        coherence_time = 0.423 / fd
        speed_rows = (
            (dataset["speed"] == speed) & dataset["valid_path"]
        )
        measured = np.percentile(dataset["fd_max"][speed_rows], 95)
        print(
            f"{speed:>3} km/h: nominal fD={fd:.1f} Hz, "
            f"Tc={coherence_time * 1e3:.2f} ms, "
            f"measured 95% |fD|={measured:.1f} Hz"
        )

plot_channel_statistics(ds)


## Step 5 — Pilot configurations

NR pilot masks are generated with `PUSCHConfig` and `PUSCHTransmitter`.
Reserved CDM resource elements are included in the pilot overhead. Two
sparser custom `PilotPattern` designs are added for comparison.


In [ ]:
import pandas as pd
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter
from sionna.phy.ofdm import PilotPattern, ResourceGrid

MCS_INDEX = 14
WIN = 96

def finish_config(name, kind, resource_grid, pusch_config=None, tx=None):
    pattern = resource_grid.pilot_pattern
    reserved = to_numpy(pattern.mask[0, 0]).astype(bool)
    pilot_vector = to_numpy(pattern.pilots[0, 0])
    pilot_grid = np.zeros((NSYM, NSC), dtype=np.complex64)
    pilot_grid[reserved] = pilot_vector
    active = np.abs(pilot_grid) > 1e-8

    pilot_symbols = np.flatnonzero(active.any(axis=1))
    pilot_subcarriers = np.flatnonzero(active.any(axis=0))
    rectangular = active[np.ix_(pilot_symbols, pilot_subcarriers)]
    if not rectangular.all() or rectangular.size != active.sum():
        raise RuntimeError(f"{name} does not form a separable pilot grid.")

    if len(pilot_symbols) > 1:
        inner_gap = int(np.diff(pilot_symbols).max())
    else:
        inner_gap = 0
    edge_gap = max(
        int(pilot_symbols[0]),
        int(NSYM - 1 - pilot_symbols[-1]),
    )
    effective_time_gap = max(inner_gap, 2 * edge_gap)
    freq_gap = int(np.diff(pilot_subcarriers).max())

    return {
        "name": name,
        "kind": kind,
        "pc": pusch_config,
        "tx": tx,
        "rg": resource_grid,
        "pilot": active,
        "reserved": reserved,
        "pilot_grid": pilot_grid,
        "Tp": pilot_symbols,
        "Fp": pilot_subcarriers,
        "Fp_w": pilot_subcarriers[pilot_subcarriers < WIN],
        "n_pilot": int(active.sum()),
        "n_reserved": int(reserved.sum()),
        "n_data": int(resource_grid.num_data_symbols),
        "eta": float(reserved.mean()),
        "df_max": freq_gap,
        "dt_eff": effective_time_gap,
    }

def make_nr_config(config_type, additional_position, num_cdm):
    pc = PUSCHConfig()
    pc.carrier.subcarrier_spacing = 30       # kHz in CarrierConfig
    pc.carrier.n_size_grid = N_PRB
    pc.n_size_bwp = N_PRB
    pc.mapping_type = "A"
    pc.symbol_allocation = [0, NSYM]
    pc.num_layers = 1
    pc.num_antenna_ports = 1
    pc.dmrs.config_type = config_type
    pc.dmrs.type_a_position = 2
    pc.dmrs.length = 1
    pc.dmrs.additional_position = additional_position
    pc.dmrs.num_cdm_groups_without_data = num_cdm
    pc.tb.mcs_table = 1
    pc.tb.mcs_index = MCS_INDEX
    pc.check_config()

    tx = PUSCHTransmitter(
        pc,
        return_bits=True,
        output_domain="freq",
        device=device,
    )
    suffix = f"-cdm{num_cdm}" if num_cdm > 1 else ""
    name = f"NR-T{config_type}-p{additional_position}{suffix}"
    return finish_config(name, "NR", tx.resource_grid, pc, tx)

def make_custom_config(spacing, symbols):
    mask = np.zeros((NSYM, NSC), dtype=bool)
    mask[np.ix_(symbols, np.arange(0, NSC, spacing))] = True
    count = int(mask.sum())
    qpsk = np.full(
        (1, 1, count),
        (1 + 1j) / np.sqrt(2),
        dtype=np.complex64,
    )
    pattern = PilotPattern(
        mask[None, None],
        qpsk,
        normalize=False,
        device=device,
    )
    grid = ResourceGrid(
        num_ofdm_symbols=NSYM,
        fft_size=NSC,
        subcarrier_spacing=SCS,
        num_tx=1,
        num_streams_per_tx=1,
        pilot_pattern=pattern,
        device=device,
    )
    name = f"comb{spacing}-{len(symbols)}sym"
    return finish_config(name, "custom", grid)

CONFIGS = [
    make_nr_config(1, 3, 1),
    make_nr_config(1, 2, 1),
    make_nr_config(1, 1, 2),
    make_nr_config(1, 1, 1),
    make_nr_config(1, 0, 1),
    make_nr_config(2, 1, 2),
    make_nr_config(2, 1, 1),
    make_nr_config(2, 0, 1),
    make_custom_config(4, [2]),
    make_custom_config(16, [2, 11]),
]
CFG = {cfg["name"]: cfg for cfg in CONFIGS}

rows = []
for cfg in CONFIGS:
    if not np.all(cfg["pilot"] <= cfg["reserved"]):
        raise RuntimeError("Active pilots must be reserved.")
    if cfg["n_data"] + cfg["n_reserved"] != NSYM * NSC:
        raise RuntimeError("Resource-grid RE count is inconsistent.")
    rows.append({
        "config": cfg["name"],
        "type": cfg["kind"],
        "pilot symbols": len(cfg["Tp"]),
        "active pilot RE": cfg["n_pilot"],
        "reserved RE": cfg["n_reserved"],
        "data RE": cfg["n_data"],
        "df max [sc]": cfg["df_max"],
        "dt effective [sym]": cfg["dt_eff"],
        "overhead [%]": round(100 * cfg["eta"], 2),
    })

PILOT_TABLE = pd.DataFrame(rows)
print(PILOT_TABLE.to_string(index=False))


In [ ]:
# Show representative masks. Blue = data, orange = reserved pilot/CDM RE.
show_names = [
    "NR-T1-p3",
    "NR-T1-p1-cdm2",
    "NR-T2-p1",
    "NR-T2-p1-cdm2",
    "comb4-1sym",
    "comb16-2sym",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), show_names):
    cfg = CFG[name]
    ax.imshow(cfg["reserved"], aspect="auto", origin="lower",
              interpolation="nearest")
    ax.set_title(f"{name}, overhead={100 * cfg['eta']:.2f}%")
    ax.set_xlabel("subcarrier")
    ax.set_ylabel("OFDM symbol")
plt.tight_layout()
plt.show()


In [ ]:
# Two-dimensional sampling check using test-region channel statistics.
# The table is a prediction, not a fitted NMSE formula.
def sampling_prediction(dataset):
    table = []
    for cfg in CONFIGS:
        for speed in SPEEDS:
            mask = (
                (dataset["split"] == "test")
                & (dataset["speed"] == speed)
                & dataset["valid_path"]
            )
            tau_max = dataset["tau_max"][mask]
            freq_ok = (
                2 * cfg["df_max"] * SCS * tau_max <= 1
            )
            time_ok = (
                2 * FD_NOM[speed] * cfg["dt_eff"] * T_SYM <= 1
            )

            freq_pass = 100 * freq_ok.mean()
            if freq_pass >= 95 and time_ok:
                verdict = "expected to interpolate well"
            elif freq_pass >= 70 and time_ok:
                verdict = "edge cases expected"
            else:
                verdict = "aliasing/interpolation loss expected"

            table.append({
                "config": cfg["name"],
                "speed [km/h]": speed,
                "overhead [%]": round(100 * cfg["eta"], 2),
                "frequency condition pass [%]": round(freq_pass, 1),
                "time condition pass": bool(time_ok),
                "prediction": verdict,
            })
    return pd.DataFrame(table)

PREDICTION = sampling_prediction(ds)
print(PREDICTION.to_string(index=False))


## Step 6 — Classical channel estimators

NR patterns use `PUSCHLSChannelEstimator` and
`PUSCHLMMSEChannelEstimator`. Custom patterns use `LSChannelEstimator`
and `LMMSEChannelEstimator`. Site covariance is estimated from the training
region, while the generic baseline uses TDL-C covariance.


In [ ]:
from sionna.phy.ofdm import tdl_freq_cov_mat, tdl_time_cov_mat

M_EVAL = 4

def view(dataset, M, split, speed=None, base_only=False):
    selected = (
        (dataset["split"] == split)
        & dataset["valid_path"]
    )
    if speed is not None:
        selected &= dataset["speed"] == speed
    if base_only:
        selected &= dataset["dir_id"] == 0
    return subarray(dataset["H"][selected], M), dataset["speed"][selected]

def view_with_locations(dataset, M, split, speed):
    selected = (
        (dataset["split"] == split)
        & (dataset["speed"] == speed)
        & (dataset["dir_id"] == 0)
        & dataset["valid_path"]
    )
    return (
        subarray(dataset["H"][selected], M),
        dataset["loc"][selected],
    )

def toeplitz_psd(matrix):
    # Average equal lags and remove small negative eigenvalues.
    n = matrix.shape[0]
    out = np.zeros_like(matrix)
    for lag in range(-(n - 1), n):
        diagonal = np.diagonal(matrix, offset=lag)
        row = np.arange(max(0, -lag), min(n, n - lag))
        out[row, row + lag] = diagonal.mean()
    out = (out + out.conj().T) / 2
    values, vectors = np.linalg.eigh(out)
    out = (vectors * np.maximum(values, 0)) @ vectors.conj().T
    return out / np.real(np.trace(out)) * n

def estimate_covariances(H):
    # H shape: [sample, antenna, time, frequency]
    n, antennas, nt, nf = H.shape
    by_frequency = H.reshape(-1, nf)
    Rf = by_frequency.T @ by_frequency.conj() / len(by_frequency)

    by_time = H.transpose(0, 1, 3, 2).reshape(-1, nt)
    Rt = by_time.T @ by_time.conj() / len(by_time)

    Rf = toeplitz_psd(Rf)
    Rt = toeplitz_psd(Rt)
    return Rf.astype(np.complex64), Rt.astype(np.complex64)

H_train, speed_train = view(ds, M_EVAL, "train")
H_test = {
    speed: view(ds, M_EVAL, "test", speed, base_only=True)[0]
    for speed in SPEEDS
}
H_validation = {
    speed: view(ds, M_EVAL, "validation", speed, base_only=True)[0]
    for speed in SPEEDS
}

Rf_site, _ = estimate_covariances(H_train)
Rt_site = {
    speed: estimate_covariances(H_train[speed_train == speed])[1]
    for speed in SPEEDS
}
COV_SITE = {
    "Rf": torch.tensor(Rf_site, device=device),
    "Rt": {
        speed: torch.tensor(Rt_site[speed], device=device)
        for speed in SPEEDS
    },
}

# True 3GPP TDL-C baseline from Sionna 2.2.
Rf_tdl = tdl_freq_cov_mat(
    "C", SCS, NSC, 300e-9, device=device
)
Rt_tdl = {
    speed: tdl_time_cov_mat(
        "C",
        speed / 3.6,
        FC,
        T_SYM,
        NSYM,
        device=device,
    )
    for speed in SPEEDS
}
COV_TDL = {"Rf": Rf_tdl, "Rt": Rt_tdl}

print("Training channels:", H_train.shape)
print("Validation channels:", {v: h.shape for v, h in H_validation.items()})
print("Test channels:", {v: h.shape for v, h in H_test.items()})
print("Site Rf/Rt:", COV_SITE["Rf"].shape, COV_SITE["Rt"][3].shape)
print("TDL-C Rf/Rt:", COV_TDL["Rf"].shape, COV_TDL["Rt"][3].shape)


In [ ]:
from sionna.phy.nr import (
    PUSCHLSChannelEstimator,
    PUSCHLMMSEChannelEstimator,
)
from sionna.phy.ofdm import (
    LSChannelEstimator,
    LMMSEChannelEstimator,
    ResourceGridMapper,
)

def make_classical_estimator(
    cfg,
    name,
    speed,
    covariances=None,
    covariance_speed=None,
):
    rg = cfg["rg"]
    covariance_speed = speed if covariance_speed is None else covariance_speed

    if cfg["kind"] == "NR":
        dmrs = cfg["pc"].dmrs
        common = {
            "resource_grid": rg,
            "dmrs_length": dmrs.length,
            "dmrs_additional_position": dmrs.additional_position,
            "num_cdm_groups_without_data": dmrs.num_cdm_groups_without_data,
            "device": device,
        }
        if name == "LS-nn":
            return PUSCHLSChannelEstimator(
                interpolation_type="nn", **common
            )
        if name == "LS-lin":
            return PUSCHLSChannelEstimator(
                interpolation_type="lin", **common
            )
        if name.startswith("LMMSE"):
            cov = COV_TDL if name == "LMMSE (TDL-C)" else (
                COV_SITE if covariances is None else covariances
            )
            return PUSCHLMMSEChannelEstimator(
                cov_mat_time=cov["Rt"][covariance_speed],
                cov_mat_freq=cov["Rf"],
                order="f-t",
                **common,
            )
    else:
        if name == "LS-nn":
            return LSChannelEstimator(
                rg, interpolation_type="nn", device=device
            )
        if name == "LS-lin":
            return LSChannelEstimator(
                rg, interpolation_type="lin", device=device
            )
        if name.startswith("LMMSE"):
            cov = COV_TDL if name == "LMMSE (TDL-C)" else (
                COV_SITE if covariances is None else covariances
            )
            return LMMSEChannelEstimator(
                rg,
                cov_mat_time=cov["Rt"][covariance_speed],
                cov_mat_freq=cov["Rf"],
                order="f-t",
                device=device,
            )
    raise KeyError(name)

def pilot_only_grid(cfg, batch_size):
    if "_grid_mapper" not in cfg:
        cfg["_grid_mapper"] = ResourceGridMapper(
            cfg["rg"], device=device
        )
    zeros = torch.zeros(
        batch_size,
        1,
        1,
        cfg["rg"].num_data_symbols,
        dtype=torch.complex64,
        device=device,
    )
    return cfg["_grid_mapper"](zeros)

def add_torch_noise(y, variance, seed):
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed))
    real = torch.randn(
        y.shape, generator=generator, device=device
    )
    imag = torch.randn(
        y.shape, generator=generator, device=device
    )
    noise = torch.complex(real, imag) * math.sqrt(variance / 2)
    return y + noise

def received_pilot_grid(H, cfg, snr_db, seed):
    H = torch.as_tensor(H, dtype=torch.complex64, device=device)
    x = pilot_only_grid(cfg, len(H))
    y = H[:, None] * x[:, :, 0].unsqueeze(2)
    noise_var = 10 ** (-snr_db / 10)
    return add_torch_noise(y, noise_var, seed), noise_var

@torch.no_grad()
def estimate_classical(
    H,
    cfg,
    estimator,
    snr_db,
    seed=SEED,
    batch_size=8,
):
    estimates = []
    error_variances = []
    for start in range(0, len(H), batch_size):
        part = H[start:start + batch_size]
        y, noise_var = received_pilot_grid(
            part, cfg, snr_db, seed + start
        )
        h_hat, err_var = estimator(
            y,
            torch.tensor(noise_var, dtype=torch.float32, device=device),
        )
        # LS error variance can be returned in a broadcastable compact
        # shape. Expand it before collecting different mini-batches.
        err_var = err_var.broadcast_to(h_hat.shape)
        estimates.append(to_numpy(h_hat[:, 0, :, 0, 0]))
        error_variances.append(to_numpy(err_var[:, 0, :, 0, 0]))
    return (
        np.concatenate(estimates).astype(np.complex64),
        np.concatenate(error_variances).astype(np.float32),
    )

def nmse_ratio(reference, estimate):
    error_energy = np.sum(np.abs(reference - estimate) ** 2)
    channel_energy = np.sum(np.abs(reference) ** 2)
    return float(error_energy / channel_energy)


In [ ]:
CLASSICAL = [
    "LS-nn",
    "LS-lin",
    "LMMSE (site)",
    "LMMSE (TDL-C)",
]
NMSE = {}

def run_classical_nmse():
    started = time.time()
    for cfg_index, cfg in enumerate(CONFIGS):
        for speed in SPEEDS:
            H = H_test[speed]
            NMSE[(cfg["name"], speed, "Perfect CSI")] = np.zeros(
                len(SNR_GRID)
            )
            for name in CLASSICAL:
                estimator = make_classical_estimator(cfg, name, speed)
                curve = []
                for snr_index, snr in enumerate(SNR_GRID):
                    h_hat, _ = estimate_classical(
                        H,
                        cfg,
                        estimator,
                        snr,
                        seed=SEED + 10_000 * cfg_index + 100 * snr_index,
                    )
                    curve.append(nmse_ratio(H, h_hat))
                NMSE[(cfg["name"], speed, name)] = np.asarray(curve)
        print(
            f"{cfg['name']:18s} finished "
            f"[{time.time() - started:.0f} s]"
        )

run_classical_nmse()


In [ ]:
def db(values):
    return 10 * np.log10(np.maximum(values, 1e-12))

def plot_nmse(speed, names, configs=CONFIGS):
    ncols = 3
    nrows = int(np.ceil(len(configs) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(14, 3.5 * nrows),
        sharex=True, sharey=True
    )
    axes = np.asarray(axes).ravel()
    for ax, cfg in zip(axes, configs):
        for name in names:
            key = (cfg["name"], speed, name)
            if key in NMSE:
                ax.plot(SNR_GRID, db(NMSE[key]), marker="o", label=name)
        ax.set_title(
            f"{cfg['name']}  (eta={100 * cfg['eta']:.2f}%)"
        )
        ax.set_xlabel("SNR [dB]")
        ax.set_ylabel("NMSE [dB]")
        ax.grid(alpha=0.3)
    for ax in axes[len(configs):]:
        ax.remove()
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(names))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    plt.show()

for speed in SPEEDS:
    plot_nmse(
        speed,
        ["LS-nn", "LS-lin", "LMMSE (site)",
         "LMMSE (TDL-C)", "Perfect CSI"],
    )

# Check whether the Step-5 sampling warning agrees with high-SNR LS.
comparison = []
high_snr_index = SNR_GRID.index(25)
for cfg in CONFIGS:
    for speed in SPEEDS:
        predicted = PREDICTION[
            (PREDICTION["config"] == cfg["name"])
            & (PREDICTION["speed [km/h]"] == speed)
        ].iloc[0]
        measured = db(
            NMSE[(cfg["name"], speed, "LS-lin")]
        )[high_snr_index]
        comparison.append({
            "config": cfg["name"],
            "speed": speed,
            "Step-5 prediction": predicted["prediction"],
            "LS-lin NMSE at 25 dB": round(float(measured), 2),
        })
print(pd.DataFrame(comparison).to_string(index=False))

print("\nSite covariance gain over TDL-C at 15 dB [dB]:")
snr_index = SNR_GRID.index(15)
rows = []
for cfg in CONFIGS:
    row = {"config": cfg["name"]}
    for speed in SPEEDS:
        site = NMSE[(cfg["name"], speed, "LMMSE (site)")][snr_index]
        tdl = NMSE[(cfg["name"], speed, "LMMSE (TDL-C)")][snr_index]
        row[f"{speed} km/h"] = round(float(db(tdl / site)), 2)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))


## Step 7 — Learned estimators

For each selected pilot pattern, I train a linear filter and a small
residual CNN. The CNN input contains the real and imaginary parts of the
interpolated LS estimate, the pilot mask and SNR. Training uses the training
region, model selection uses validation MSE, and final NMSE is measured on
the test region.

The CNN is trained with MSE. Speed is not included as an input so that the
same model can be evaluated under speed mismatch.


In [ ]:
def interpolation_matrices(cfg, kind="lin"):
    def one_dimension(nodes, size):
        result = np.zeros((size, len(nodes)), dtype=np.float32)
        x = np.arange(size)
        if kind == "nn":
            nearest = np.argmin(
                np.abs(x[:, None] - nodes[None, :]), axis=1
            )
            result[x, nearest] = 1.0
        else:
            for index in range(len(nodes)):
                basis = np.zeros(len(nodes))
                basis[index] = 1.0
                result[:, index] = np.interp(x, nodes, basis)
        return result
    return (
        one_dimension(cfg["Tp"], NSYM),
        one_dimension(cfg["Fp"], NSC),
    )

def pilot_samples(H, cfg):
    return H[:, :, cfg["Tp"]][:, :, :, cfg["Fp"]]

def active_pilot_values(cfg):
    return cfg["pilot_grid"][np.ix_(cfg["Tp"], cfg["Fp"])]

def interpolate_ls(noisy_ls, cfg):
    At, Af = interpolation_matrices(cfg, "lin")
    return np.einsum(
        "tp,...pf,kf->...tk",
        At,
        noisy_ls,
        Af,
        optimize=True,
    )

def noisy_ls_samples(H, cfg, noise_var, rng):
    clean = pilot_samples(H, cfg)
    pilots = active_pilot_values(cfg)
    noise = complex_noise(clean.shape, noise_var, rng)
    return clean + noise / pilots

def to_windows(H):
    n, antennas, nt, nf = H.shape
    return (
        H.reshape(n * antennas, nt, nf // WIN, WIN)
        .transpose(0, 2, 1, 3)
        .reshape(-1, nt, WIN)
    )

def run_windowed(function, noisy_pilots, cfg):
    num_windows = NSC // WIN
    num_time = len(cfg["Tp"])
    num_pilots_window = len(cfg["Fp_w"])
    leading = noisy_pilots.shape[:-2]

    reshaped = noisy_pilots.reshape(
        *leading, num_time, num_windows, num_pilots_window
    )
    reshaped = np.moveaxis(reshaped, -2, -3)
    rows = reshaped.reshape(-1, num_time, num_pilots_window)
    estimated = function(rows)
    estimated = estimated.reshape(
        *leading, num_windows, NSYM, WIN
    )
    estimated = np.moveaxis(estimated, -3, -2)
    return estimated.reshape(*leading, NSYM, NSC)

def fit_ridge(
    train_windows,
    validation_windows,
    cfg,
    noise_var,
    seed,
    lambdas=(1e-4, 1e-3, 1e-2, 1e-1),
):
    rng = np.random.default_rng(seed)
    clean_p = train_windows[:, cfg["Tp"]][:, :, cfg["Fp_w"]]
    B = len(train_windows)
    L = clean_p.shape[1] * clean_p.shape[2]
    target = train_windows.reshape(B, -1)

    # Pilot amplitudes are constant within these NR/custom patterns.
    pilot_amp = np.abs(active_pilot_values(cfg)[:, :len(cfg["Fp_w"])])
    noisy = clean_p + complex_noise(clean_p.shape, noise_var, rng) / pilot_amp
    observations = noisy.reshape(B, L)
    Ryy = observations.T @ observations.conj() / B
    Rhy = target.T @ observations.conj() / B
    scale = np.trace(Ryy).real / L

    def solve(lam):
        identity = np.eye(L, dtype=np.complex128)
        return (
            Rhy @ np.linalg.solve(Ryy + lam * scale * identity, identity)
        ).astype(np.complex64)

    clean_val = validation_windows[:, cfg["Tp"]][
        :, :, cfg["Fp_w"]
    ]
    val_rng = np.random.default_rng(seed + 1)
    noisy_val = (
        clean_val
        + complex_noise(clean_val.shape, noise_var, val_rng) / pilot_amp
    )
    noisy_val = noisy_val.reshape(len(validation_windows), L)

    candidates = [solve(lam) for lam in lambdas]
    losses = []
    for matrix in candidates:
        prediction = noisy_val @ matrix.T
        losses.append(np.mean(
            np.abs(prediction - validation_windows.reshape(
                len(validation_windows), -1
            )) ** 2
        ))
    return candidates[int(np.argmin(losses))]

def fit_all_ridge(cfg, train_speeds=SPEEDS):
    filters = {}
    for speed in train_speeds:
        train_windows = to_windows(H_train[speed_train == speed])
        validation_windows = to_windows(H_validation[speed])
        for snr in SNR_GRID:
            filters[(snr, speed)] = fit_ridge(
                train_windows,
                validation_windows,
                cfg,
                10 ** (-snr / 10),
                seed=SEED + speed + snr,
            )
    return filters

def ridge_estimator(filters, cfg, design_speed):
    def estimate(noisy_pilots, noise_var):
        snr = int(round(-10 * np.log10(noise_var)))
        matrix = filters[(snr, design_speed)]
        return run_windowed(
            lambda rows: (
                rows.reshape(len(rows), -1) @ matrix.T
            ).reshape(-1, NSYM, WIN),
            noisy_pilots,
            cfg,
        )
    return estimate

def learned_nmse_curve(H, cfg, estimator, repetitions=1, seed=SEED):
    denominator = np.sum(np.abs(H) ** 2)
    curve = []
    for snr_index, snr in enumerate(SNR_GRID):
        noise_var = 10 ** (-snr / 10)
        error = 0.0
        for repeat in range(repetitions):
            rng = np.random.default_rng(
                seed + 100 * snr_index + repeat
            )
            noisy = noisy_ls_samples(H, cfg, noise_var, rng)
            h_hat = estimator(noisy, noise_var)
            error += np.sum(np.abs(H - h_hat) ** 2)
        curve.append(error / (repetitions * denominator))
    return np.asarray(curve)


In [ ]:
import gc
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, channels=24):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(4, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=(1, 2),
                      dilation=(1, 2)),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=(1, 4),
                      dilation=(1, 4)),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=(1, 8),
                      dilation=(1, 8)),
            nn.ReLU(),
        )
        self.output = nn.Conv2d(channels, 2, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, ls_grid, mask, snr_db):
        batch, _, nt, nf = ls_grid.shape
        snr = torch.as_tensor(
            snr_db, dtype=torch.float32, device=ls_grid.device
        )
        if snr.numel() == 1:
            snr = snr.reshape(1, 1, 1, 1).expand(
                batch, 1, nt, nf
            )
        else:
            snr = snr.reshape(batch, 1, 1, 1).expand(
                batch, 1, nt, nf
            )
        x = torch.cat([
            ls_grid,
            mask.expand(batch, -1, -1, -1),
            snr / 30.0,
        ], dim=1)
        return ls_grid + self.output(self.body(x))

def train_cnn(
    cfg,
    steps=1200,
    batch_size=32,
    train_speeds=SPEEDS,
    seed=SEED,
    validation_every=200,
):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    Tp = torch.tensor(cfg["Tp"], dtype=torch.long, device=device)
    Fp = torch.tensor(cfg["Fp"], dtype=torch.long, device=device)
    At_np, Af_np = interpolation_matrices(cfg, "lin")
    At = torch.tensor(At_np, dtype=torch.float32, device=device)
    Af = torch.tensor(Af_np, dtype=torch.float32, device=device)
    pilot_values = torch.tensor(
        active_pilot_values(cfg),
        dtype=torch.complex64,
        device=device,
    )
    mask = torch.tensor(
        cfg["pilot"].astype(np.float32), device=device
    )[None, None]

    def flatten_antennas(H):
        return torch.tensor(
            H.reshape(-1, NSYM, NSC),
            dtype=torch.complex64,
            device=device,
        )

    train_data = {
        speed: flatten_antennas(H_train[speed_train == speed])
        for speed in train_speeds
    }
    validation_data = {
        speed: flatten_antennas(H_validation[speed])
        for speed in train_speeds
    }

    def make_input(H, snr_db, fixed_noise=None):
        clean = H[:, Tp][:, :, Fp]
        if fixed_noise is None:
            real = torch.randn(clean.shape, device=device)
            imag = torch.randn(clean.shape, device=device)
            noise = torch.complex(real, imag) / math.sqrt(2)
        else:
            noise = fixed_noise
        noisy = (
            clean
            + noise * math.sqrt(10 ** (-snr_db / 10))
            / pilot_values
        )
        real_grid = At @ noisy.real @ Af.T
        imag_grid = At @ noisy.imag @ Af.T
        return torch.stack([real_grid, imag_grid], dim=1)

    validation_noise = {}
    generator = torch.Generator(device=device)
    generator.manual_seed(seed + 99)
    for speed in train_speeds:
        count = len(validation_data[speed])
        for snr in SNR_GRID:
            shape = (count, len(cfg["Tp"]), len(cfg["Fp"]))
            real = torch.randn(
                shape, generator=generator, device=device
            )
            imag = torch.randn(
                shape, generator=generator, device=device
            )
            validation_noise[(speed, snr)] = (
                torch.complex(real, imag) / math.sqrt(2)
            )

    model = SmallCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=2e-3, weight_decay=1e-5
    )

    def validation_mse():
        model.eval()
        values = []
        with torch.no_grad():
            for speed in train_speeds:
                target_complex = validation_data[speed]
                target = torch.stack([
                    target_complex.real, target_complex.imag
                ], dim=1)
                for snr in SNR_GRID:
                    ls = make_input(
                        target_complex,
                        snr,
                        validation_noise[(speed, snr)],
                    )
                    prediction = model(ls, mask, snr)
                    values.append(
                        torch.mean((prediction - target) ** 2).item()
                    )
        model.train()
        return float(np.mean(values))

    best_loss = validation_mse()
    best_state = {
        key: value.detach().clone()
        for key, value in model.state_dict().items()
    }
    print(
        f"{cfg['name']}: step 0 validation MSE "
        f"{10 * np.log10(best_loss):.2f} dB"
    )

    model.train()
    for step in range(1, steps + 1):
        speed = int(rng.choice(train_speeds))
        snr = float(rng.uniform(-5, 25))
        data = train_data[speed]
        indices = torch.tensor(
            rng.integers(0, len(data), size=batch_size),
            device=device,
        )
        H = data[indices]
        crop_start = int(rng.choice([0, 96, 192]))

        ls = make_input(H, snr)
        target = torch.stack([H.real, H.imag], dim=1)
        ls = ls[..., crop_start:crop_start + WIN]
        target = target[..., crop_start:crop_start + WIN]
        mask_crop = mask[..., crop_start:crop_start + WIN]

        prediction = model(ls, mask_crop, snr)
        loss = torch.mean((prediction - target) ** 2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step == steps // 2:
            for group in optimizer.param_groups:
                group["lr"] *= 0.3

        if step % validation_every == 0 or step == steps:
            score = validation_mse()
            if score < best_loss:
                best_loss = score
                best_state = {
                    key: value.detach().clone()
                    for key, value in model.state_dict().items()
                }
            print(
                f"{cfg['name']}: step {step:4d}, "
                f"train MSE {10 * np.log10(loss.item()):.2f} dB, "
                f"validation MSE {10 * np.log10(score):.2f} dB"
            )

    model.load_state_dict(best_state)
    model.eval()
    return model

def cnn_estimator(model, cfg, inference_batch=64):
    mask = torch.tensor(
        cfg["pilot"].astype(np.float32), device=device
    )[None, None]

    def estimate(noisy_pilots, noise_var):
        ls = interpolate_ls(noisy_pilots, cfg)
        leading = ls.shape[:-2]
        flat = ls.reshape(-1, NSYM, NSC)
        output = []
        snr_db = -10 * np.log10(noise_var)
        with torch.no_grad():
            for start in range(0, len(flat), inference_batch):
                part = flat[start:start + inference_batch]
                x = torch.tensor(
                    np.stack([part.real, part.imag], axis=1),
                    dtype=torch.float32,
                    device=device,
                )
                prediction = model(x, mask, snr_db)
                output.append(torch.complex(
                    prediction[:, 0], prediction[:, 1]
                ).cpu().numpy())
        return np.concatenate(output).reshape(
            *leading, NSYM, NSC
        ).astype(np.complex64)
    return estimate

class LearnedGridEstimator(nn.Module):
    # Adapter used later by PUSCHReceiver and LMMSEEqualizer.
    def __init__(self, cfg, model, error_variance):
        super().__init__()
        self.model = model
        self.error_variance = float(error_variance)
        self.register_buffer(
            "Tp", torch.tensor(cfg["Tp"], dtype=torch.long)
        )
        self.register_buffer(
            "Fp", torch.tensor(cfg["Fp"], dtype=torch.long)
        )
        At, Af = interpolation_matrices(cfg, "lin")
        self.register_buffer(
            "At", torch.tensor(At, dtype=torch.float32)
        )
        self.register_buffer(
            "Af", torch.tensor(Af, dtype=torch.float32)
        )
        self.register_buffer(
            "pilots",
            torch.tensor(
                active_pilot_values(cfg), dtype=torch.complex64
            ),
        )
        self.register_buffer(
            "mask",
            torch.tensor(
                cfg["pilot"].astype(np.float32)
            )[None, None],
        )

    def forward(self, y, no):
        # y: [batch, rx=1, antenna, time, frequency]
        received = y[:, 0]
        noisy_ls = received[:, :, self.Tp][:, :, :, self.Fp]
        noisy_ls = noisy_ls / self.pilots

        batch, antennas = noisy_ls.shape[:2]
        rows = noisy_ls.reshape(
            batch * antennas, len(self.Tp), len(self.Fp)
        )
        real_grid = self.At @ rows.real @ self.Af.T
        imag_grid = self.At @ rows.imag @ self.Af.T
        ls = torch.stack([real_grid, imag_grid], dim=1)

        snr_db = -10 * torch.log10(torch.as_tensor(
            no, dtype=torch.float32, device=y.device
        ).mean())
        prediction = self.model(ls, self.mask, snr_db)
        h_hat = torch.complex(
            prediction[:, 0], prediction[:, 1]
        ).reshape(batch, antennas, NSYM, NSC)
        h_hat = h_hat[:, None, :, None, None]
        err_var = torch.full_like(
            h_hat.real, self.error_variance
        )
        return h_hat, err_var


In [ ]:
LEARN_CONFIGS = [
    "NR-T1-p3",
    "NR-T1-p1",
    "NR-T1-p0",
    "NR-T2-p0",
    "comb4-1sym",
    "comb16-2sym",
]
FILTERS = {}
CNN = {}
STEPS_CNN = 1200

for name in LEARN_CONFIGS:
    cfg = CFG[name]
    print("\nTraining learned linear estimator:", name)
    FILTERS[name] = fit_all_ridge(cfg)
    for speed in SPEEDS:
        estimator = ridge_estimator(FILTERS[name], cfg, speed)
        NMSE[(name, speed, "Ridge (learned linear)")] = (
            learned_nmse_curve(H_test[speed], cfg, estimator)
        )

    print("Training CNN:", name)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    CNN[name] = train_cnn(cfg, steps=STEPS_CNN)
    estimator = cnn_estimator(CNN[name], cfg)
    for speed in SPEEDS:
        NMSE[(name, speed, "CNN (learned)")] = (
            learned_nmse_curve(H_test[speed], cfg, estimator)
        )

torch.save(
    {
        "ridge": FILTERS,
        "site_metadata": site_a_meta,
        "cnn": {
            name: model.state_dict()
            for name, model in CNN.items()
        },
    },
    "learned_estimators.pt",
)


In [ ]:
LEARNED_NAMES = [
    "LS-lin",
    "LMMSE (site)",
    "LMMSE (TDL-C)",
    "Ridge (learned linear)",
    "CNN (learned)",
]
selected_configs = [CFG[name] for name in LEARN_CONFIGS]
for speed in SPEEDS:
    plot_nmse(speed, LEARNED_NAMES, selected_configs)

snr_index = SNR_GRID.index(15)
summary = []
for name in LEARN_CONFIGS:
    for speed in SPEEDS:
        row = {"config": name, "speed": speed}
        for estimator_name in LEARNED_NAMES:
            row[estimator_name] = round(float(db(
                NMSE[(name, speed, estimator_name)][snr_index]
            )), 2)
        summary.append(row)
print("NMSE at 15 dB [dB]")
print(pd.DataFrame(summary).to_string(index=False))


In [ ]:
# Same-device inference timing: both estimators receive an already-created
# PyTorch resource grid on the selected device.
def time_estimator(function, y, no, repetitions=10):
    with torch.no_grad():
        function(y, no)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        started = time.perf_counter()
        for _ in range(repetitions):
            function(y, no)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - started) / (
        repetitions * len(y)
    )

timing_rows = []
for name in LEARN_CONFIGS:
    cfg = CFG[name]
    sample = H_validation[120][:8]
    y, noise_var = received_pilot_grid(sample, cfg, 15, SEED)

    lmmse_est = make_classical_estimator(
        cfg, "LMMSE (site)", 120
    )
    cnn_est = LearnedGridEstimator(
        cfg, CNN[name], error_variance=0.0
    ).to(device)

    timing_rows.append({
        "config": name,
        "LMMSE stored real values": int(
            2 * (NSC * NSC + NSYM * NSYM)
        ),
        "CNN parameters": int(sum(
            p.numel() for p in CNN[name].parameters()
        )),
        "LMMSE ms/grid": round(time_estimator(
            lmmse_est, y, torch.tensor(noise_var, device=device)
        ), 4),
        "CNN ms/grid": round(time_estimator(
            cnn_est, y, torch.tensor(noise_var, device=device)
        ), 4),
    })

print("Timing device:", device)
print(pd.DataFrame(timing_rows).to_string(index=False))


## Step 8 — Link-level impact

NR configurations use `PUSCHTransmitter`, the ray-traced channel and
`PUSCHReceiver` with LMMSE detection and LDPC decoding.

The custom configuration uses `TBEncoder`, `Mapper`, `ResourceGridMapper`,
the channel estimator, `LMMSEEqualizer`, `Demapper` and `TBDecoder`. For the
CNN, validation MSE is used as the channel-estimation error variance.


In [ ]:
from sionna.phy.nr import (
    PUSCHReceiver,
    TBEncoder,
    TBDecoder,
    calculate_tb_size,
)
from sionna.phy.mapping import BinarySource, Mapper, Demapper
from sionna.phy.mimo import StreamManagement
from sionna.phy.ofdm import LMMSEEqualizer

_VALIDATION_MSE = {}

def validation_mse_cnn(cfg, speed, snr, M):
    key = (cfg["name"], speed, snr, M)
    if key not in _VALIDATION_MSE:
        H, _ = view(ds, M, "validation", speed, base_only=True)
        rng = np.random.default_rng(
            SEED + 1000 * M + 10 * speed + snr
        )
        noise_var = 10 ** (-snr / 10)
        noisy = noisy_ls_samples(H, cfg, noise_var, rng)
        h_hat = cnn_estimator(CNN[cfg["name"]], cfg)(
            noisy, noise_var
        )
        _VALIDATION_MSE[key] = float(
            np.mean(np.abs(H - h_hat) ** 2)
        )
    return _VALIDATION_MSE[key]

def make_link_estimator(cfg, name, speed, snr, M):
    if name == "Perfect CSI":
        return None
    if name in ("LS-lin", "LMMSE (site)"):
        return make_classical_estimator(cfg, name, speed)
    if name == "CNN (learned)":
        mse = validation_mse_cnn(cfg, speed, snr, M)
        return LearnedGridEstimator(
            cfg, CNN[cfg["name"]], mse
        ).to(device)
    raise KeyError(name)

class NRLink:
    def __init__(self, cfg, estimator):
        self.cfg = cfg
        self.tx = cfg["tx"]
        self.perfect = estimator is None
        receiver_estimator = "perfect" if self.perfect else estimator
        self.rx = PUSCHReceiver(
            self.tx,
            channel_estimator=receiver_estimator,
            input_domain="freq",
            return_tb_crc_status=True,
            device=device,
        )
        self.tbs = int(self.tx._tb_size)

    @torch.no_grad()
    def run_batch(self, H, snr_db, seed):
        batch_size = len(H)
        sionna_config.seed = int(seed)
        torch.manual_seed(int(seed))
        x, bits = self.tx(batch_size)

        y = H[:, None] * x[:, :, 0].unsqueeze(2)
        noise_var = 10 ** (-snr_db / 10)
        y = add_torch_noise(y, noise_var, seed + 1)
        no = torch.tensor(
            noise_var, dtype=torch.float32, device=device
        )

        if self.perfect:
            h_true = H[:, None, :, None, None]
            bits_hat, crc_ok = self.rx(y, no, h_true)
        else:
            bits_hat, crc_ok = self.rx(y, no)

        block_error = torch.any(bits_hat != bits, dim=-1)
        return int(block_error.sum().item()), int(crc_ok.numel())

class CustomLink:
    def __init__(self, cfg, estimator, qm=4, rate=0.5):
        self.cfg = cfg
        self.estimator = estimator
        self.perfect = estimator is None
        self.qm = qm
        self.rate = rate

        rg = cfg["rg"]
        num_coded_bits = int(rg.num_data_symbols * qm)
        tbs = calculate_tb_size(
            modulation_order=qm,
            target_coderate=rate,
            num_coded_bits=num_coded_bits,
            num_layers=1,
            device=device,
        )[0]
        self.tbs = int(tbs.item())

        self.source = BinarySource(device=device)
        self.encoder = TBEncoder(
            target_tb_size=self.tbs,
            num_coded_bits=num_coded_bits,
            target_coderate=rate,
            num_bits_per_symbol=qm,
            device=device,
        )
        self.decoder = TBDecoder(
            self.encoder, num_bp_iter=20, device=device
        )
        self.mapper = Mapper("qam", qm, device=device)
        self.grid_mapper = ResourceGridMapper(rg, device=device)
        stream_management = StreamManagement(
            np.ones((1, 1), dtype=bool), 1
        )
        self.equalizer = LMMSEEqualizer(
            rg, stream_management, device=device
        )
        self.demapper = Demapper(
            "app", "qam", qm, device=device
        )

    @torch.no_grad()
    def run_batch(self, H, snr_db, seed):
        batch_size = len(H)
        sionna_config.seed = int(seed)
        torch.manual_seed(int(seed))

        bits = self.source([batch_size, 1, self.tbs])
        coded = self.encoder(bits)
        symbols = self.mapper(coded).unsqueeze(2)
        x = self.grid_mapper(symbols)

        y = H[:, None] * x[:, :, 0].unsqueeze(2)
        noise_var = 10 ** (-snr_db / 10)
        y = add_torch_noise(y, noise_var, seed + 1)
        no = torch.tensor(
            noise_var, dtype=torch.float32, device=device
        )

        if self.perfect:
            h_hat = H[:, None, :, None, None]
            err_var = torch.zeros(
                1, dtype=torch.float32, device=device
            )
        else:
            h_hat, err_var = self.estimator(y, no)

        x_hat, no_eff = self.equalizer(
            y, h_hat, err_var, no
        )
        llr = self.demapper(x_hat, no_eff).squeeze(2)
        bits_hat, crc_ok = self.decoder(llr)

        block_error = torch.any(bits_hat != bits, dim=-1)
        return int(block_error.sum().item()), int(crc_ok.numel())

def make_link(cfg, estimator_name, speed, snr, M):
    estimator = make_link_estimator(
        cfg, estimator_name, speed, snr, M
    )
    if cfg["kind"] == "NR":
        return NRLink(cfg, estimator)
    return CustomLink(cfg, estimator)


In [ ]:
LINK_CONFIGS = [
    "NR-T1-p3",
    "NR-T1-p1",
    "NR-T1-p0",
    "comb4-1sym",
]
LINK_SPEEDS = [3, 120]
LINK_ESTIMATORS = [
    "Perfect CSI",
    "LS-lin",
    "LMMSE (site)",
    "CNN (learned)",
]
LINK_SNR = [0, 5, 10, 15, 20, 25]
M_LINK = 2

# Set this to False for a short receiver test.
FULL_LINK_RUN = True
if FULL_LINK_RUN:
    MAX_SLOTS, MIN_ERRORS, LINK_BATCH = 2500, 150, 25
else:
    MAX_SLOTS, MIN_ERRORS, LINK_BATCH = 300, 20, 20

BLER = {}
BLER_BLOCKS = {}
BLER_ERRORS = {}
TBS = {}

def point_seed(cfg_name, speed, snr, batch_index):
    cfg_index = [c["name"] for c in CONFIGS].index(cfg_name)
    speed_index = SPEEDS.index(speed)
    snr_index = LINK_SNR.index(snr)
    return (
        SEED
        + 1_000_000 * cfg_index
        + 10_000 * speed_index
        + 100 * snr_index
        + batch_index
    )

def run_bler_point(cfg, speed, estimator_name, snr, M):
    H_np = view(ds, M, "test", speed, base_only=True)[0]
    H = torch.tensor(H_np, dtype=torch.complex64, device=device)
    link = make_link(cfg, estimator_name, speed, snr, M)

    # This RNG does not depend on the estimator name. Estimators therefore
    # see the same sequence of channel indices, bits and noise.
    seed0 = point_seed(cfg["name"], speed, snr, 0)
    rng = np.random.default_rng(seed0)
    errors = 0
    blocks = 0
    batch_index = 0

    while blocks < MAX_SLOTS and errors < MIN_ERRORS:
        this_batch = min(LINK_BATCH, MAX_SLOTS - blocks)
        indices = rng.integers(0, len(H), size=this_batch)
        batch_H = H[torch.tensor(indices, device=device)]
        seed = point_seed(
            cfg["name"], speed, snr, batch_index
        )
        batch_errors, batch_blocks = link.run_batch(
            batch_H, snr, seed
        )
        errors += batch_errors
        blocks += batch_blocks
        batch_index += 1

    return errors / blocks, errors, blocks, link.tbs

def run_link_sweep(
    config_names=LINK_CONFIGS,
    speeds=LINK_SPEEDS,
    estimator_names=LINK_ESTIMATORS,
    snrs=LINK_SNR,
    M=M_LINK,
):
    started = time.time()
    for config_name in config_names:
        cfg = CFG[config_name]
        for speed in speeds:
            for estimator_name in estimator_names:
                values = []
                counts = []
                errors = []
                for snr in snrs:
                    value, n_error, n_block, tbs = run_bler_point(
                        cfg, speed, estimator_name, snr, M
                    )
                    values.append(value)
                    errors.append(n_error)
                    counts.append(n_block)
                    TBS[config_name] = tbs

                key = (estimator_name, config_name, speed)
                BLER[key] = np.asarray(values)
                BLER_ERRORS[key] = np.asarray(errors)
                BLER_BLOCKS[key] = np.asarray(counts)
                print(
                    f"{config_name:12s} {speed:>3} km/h "
                    f"{estimator_name:14s} "
                    f"BLER={np.round(values, 3)} "
                    f"blocks={counts} "
                    f"[{time.time() - started:.0f} s]"
                )

run_link_sweep()
torch.save(
    {
        "bler": BLER,
        "blocks": BLER_BLOCKS,
        "errors": BLER_ERRORS,
        "tbs": TBS,
        "site_metadata": site_a_meta,
    },
    "link_results.pt",
)


In [ ]:
def plot_bler(config_names=LINK_CONFIGS, speeds=LINK_SPEEDS):
    fig, axes = plt.subplots(
        len(speeds),
        len(config_names),
        figsize=(4 * len(config_names), 3.2 * len(speeds)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for row, speed in enumerate(speeds):
        for col, config_name in enumerate(config_names):
            ax = axes[row, col]
            for estimator_name in LINK_ESTIMATORS:
                key = (estimator_name, config_name, speed)
                if key not in BLER:
                    continue
                observed_floor = 0.5 / BLER_BLOCKS[key]
                values = np.maximum(BLER[key], observed_floor)
                ax.semilogy(
                    LINK_SNR,
                    values,
                    marker="o",
                    label=estimator_name,
                )
            ax.axhline(0.1, color="black", linestyle=":")
            ax.set_title(
                f"{config_name}\neta={100 * CFG[config_name]['eta']:.2f}%"
            )
            ax.set_xlabel("SNR [dB]")
            ax.set_ylabel(f"{speed} km/h BLER")
            ax.grid(alpha=0.3)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

plot_bler()


In [ ]:
def estimator_grid_for_rate(
    H,
    cfg,
    estimator_name,
    speed,
    snr,
    M,
    seed=SEED,
):
    if estimator_name == "Perfect CSI":
        return H, np.zeros(H.shape, dtype=np.float32)

    estimator = make_link_estimator(
        cfg, estimator_name, speed, snr, M
    )
    if estimator_name in ("LS-lin", "LMMSE (site)"):
        return estimate_classical(
            H, cfg, estimator, snr, seed=seed
        )

    estimates = []
    variances = []
    with torch.no_grad():
        for start in range(0, len(H), 8):
            part = H[start:start + 8]
            y, noise_var = received_pilot_grid(
                part, cfg, snr, seed + start
            )
            h_hat, err_var = estimator(
                y,
                torch.tensor(noise_var, device=device),
            )
            estimates.append(to_numpy(h_hat[:, 0, :, 0, 0]))
            variances.append(to_numpy(err_var[:, 0, :, 0, 0]))
    return np.concatenate(estimates), np.concatenate(variances)

def actual_rate_bound(cfg, estimator_name, speed, snr, M=M_LINK):
    # Evaluate the rate using the simulated channel estimates.
    H = view(ds, M, "test", speed, base_only=True)[0]
    h_hat, err_var = estimator_grid_for_rate(
        H, cfg, estimator_name, speed, snr, M
    )
    signal_power = np.sum(np.abs(h_hat) ** 2, axis=1)
    error_power = np.sum(err_var, axis=1)
    rho = 10 ** (snr / 10)
    sinr = rho * signal_power / (1 + rho * error_power)
    return float(
        (1 - cfg["eta"]) * np.mean(np.log2(1 + sinr))
    )

RATE_BOUND = {}
for config_name in LINK_CONFIGS:
    cfg = CFG[config_name]
    for speed in LINK_SPEEDS:
        for estimator_name in LINK_ESTIMATORS:
            for snr in LINK_SNR:
                RATE_BOUND[
                    (estimator_name, config_name, speed, snr)
                ] = actual_rate_bound(
                    cfg, estimator_name, speed, snr
                )

def modulation_and_rate(cfg):
    if cfg["kind"] == "NR":
        return (
            int(cfg["pc"].tb.num_bits_per_symbol),
            float(cfg["pc"].tb.target_coderate),
        )
    return 4, 0.5

def goodput_tbs(config_name, estimator_name, speed):
    return (
        TBS[config_name]
        * (1 - BLER[(estimator_name, config_name, speed)])
        / T_SLOT
        / 1e6
    )

def goodput_data_re(config_name, estimator_name, speed):
    cfg = CFG[config_name]
    qm, rate = modulation_and_rate(cfg)
    return (
        qm
        * rate
        * cfg["n_data"]
        * (1 - BLER[(estimator_name, config_name, speed)])
        / T_SLOT
        / 1e6
    )

def eta_star_table():
    rows = []
    for speed in LINK_SPEEDS:
        for snr_index, snr in enumerate(LINK_SNR):
            for estimator_name in LINK_ESTIMATORS:
                gp_re = {
                    name: goodput_data_re(
                        name, estimator_name, speed
                    )[snr_index]
                    for name in LINK_CONFIGS
                }
                gp_tbs = {
                    name: goodput_tbs(
                        name, estimator_name, speed
                    )[snr_index]
                    for name in LINK_CONFIGS
                }
                rates = {
                    name: RATE_BOUND[
                        (estimator_name, name, speed, snr)
                    ]
                    for name in LINK_CONFIGS
                }

                def best_eta(values):
                    best = max(
                        values,
                        key=lambda name: (
                            values[name], -CFG[name]["eta"]
                        ),
                    )
                    return round(100 * CFG[best]["eta"], 2)

                rows.append({
                    "speed": speed,
                    "SNR": snr,
                    "estimator": estimator_name,
                    "eta* goodput data-RE [%]": best_eta(gp_re),
                    "eta* goodput TBS [%]": best_eta(gp_tbs),
                    "eta* actual R(eta) [%]": best_eta(rates),
                })
    return pd.DataFrame(rows)

ETA_STAR = eta_star_table()
print(ETA_STAR.to_string(index=False))

print("\nTransport block sizes:")
print(pd.DataFrame([
    {
        "config": name,
        "overhead [%]": round(100 * CFG[name]["eta"], 2),
        "data RE": CFG[name]["n_data"],
        "TBS": TBS[name],
    }
    for name in LINK_CONFIGS
]).to_string(index=False))
print(
    "\nNR TBS can stay equal for different pilot overheads because "
    "TS 38.214 caps usable REs per PRB at 156."
)


In [ ]:
def plot_goodput_and_rate(snrs=(10, 20)):
    for speed in LINK_SPEEDS:
        fig, axes = plt.subplots(
            2, len(snrs), figsize=(6 * len(snrs), 7),
            squeeze=False
        )
        ordered = sorted(
            LINK_CONFIGS, key=lambda name: CFG[name]["eta"]
        )
        overhead = [100 * CFG[name]["eta"] for name in ordered]
        for col, snr in enumerate(snrs):
            snr_index = LINK_SNR.index(snr)
            for estimator_name in LINK_ESTIMATORS:
                goodput = [
                    goodput_data_re(
                        name, estimator_name, speed
                    )[snr_index]
                    for name in ordered
                ]
                rate = [
                    RATE_BOUND[
                        (estimator_name, name, speed, snr)
                    ]
                    * NSYM * NSC / T_SLOT / 1e6
                    for name in ordered
                ]
                axes[0, col].plot(
                    overhead, goodput, marker="o",
                    label=estimator_name
                )
                axes[1, col].plot(
                    overhead, rate, marker="o",
                    label=estimator_name
                )

            axes[0, col].set_title(
                f"{speed} km/h, SNR={snr} dB"
            )
            axes[0, col].set_ylabel("Goodput [Mbit/s]")
            axes[1, col].set_ylabel("Actual R(eta) [Mbit/s]")
            axes[1, col].set_xlabel("Pilot overhead [%]")
            axes[0, col].grid(alpha=0.3)
            axes[1, col].grid(alpha=0.3)

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=4)
        fig.tight_layout(rect=(0, 0.06, 1, 1))
        plt.show()

plot_goodput_and_rate()


### Effect of the number of BS antennas

Each BS antenna estimates its channel from the same pilot resource elements.
More antennas provide array gain, while channel-estimation errors are
included by the error-aware LMMSE detector. The same receiver chain is used
for M=1, 2 and 4.


In [ ]:
M_CONFIG = "NR-T1-p1"
M_VALUES = [1, 2, 4]
M_SPEEDS = [3, 120]
M_SNR = [0, 5, 10, 15, 20]
M_ESTIMATORS = ["LS-lin", "LMMSE (site)", "CNN (learned)"]
BLER_M = {}

old_max, old_min = MAX_SLOTS, MIN_ERRORS
MAX_SLOTS, MIN_ERRORS = 1200, 100
for speed in M_SPEEDS:
    for M in M_VALUES:
        for estimator_name in M_ESTIMATORS:
            values = []
            for snr in M_SNR:
                value, _, _, _ = run_bler_point(
                    CFG[M_CONFIG], speed, estimator_name, snr, M
                )
                values.append(value)
            BLER_M[(M, estimator_name, speed)] = np.asarray(values)
            print(
                f"{speed:>3} km/h, M={M}, "
                f"{estimator_name}: {np.round(values, 3)}"
            )
MAX_SLOTS, MIN_ERRORS = old_max, old_min

fig, axes = plt.subplots(
    len(M_SPEEDS), len(M_ESTIMATORS),
    figsize=(13, 6), sharex=True, sharey=True
)
for row, speed in enumerate(M_SPEEDS):
    for col, estimator_name in enumerate(M_ESTIMATORS):
        ax = axes[row, col]
        for M in M_VALUES:
            ax.semilogy(
                M_SNR,
                np.maximum(
                    BLER_M[(M, estimator_name, speed)],
                    0.5 / 1200,
                ),
                marker="o",
                label=f"M={M} ({M*M} antennas)",
            )
        ax.axhline(0.1, color="black", linestyle=":")
        ax.set_title(estimator_name)
        ax.set_xlabel("SNR [dB]")
        ax.set_ylabel(f"{speed} km/h BLER")
        ax.grid(alpha=0.3)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()


## Step 9 — Reduced pilots and mismatch

Pilot saving is evaluated at BLER <= 10%. Speed mismatch compares models
trained at 3 and 120 km/h. Site mismatch uses a second BS position; its
matched covariance is estimated from the Site-B training split and evaluated
on the Site-B test split.


In [ ]:
REDUCED_CONFIGS = [
    "NR-T1-p3",
    "NR-T1-p1",
    "NR-T1-p0",
    "comb4-1sym",
    "comb16-2sym",
]
TARGET_SNR = 15
TARGET_BLER = 0.10

missing_configs = []
for name in REDUCED_CONFIGS:
    complete = all(
        (estimator, name, speed) in BLER
        for speed in LINK_SPEEDS
        for estimator in LINK_ESTIMATORS
    )
    if not complete:
        missing_configs.append(name)
if missing_configs:
    run_link_sweep(config_names=missing_configs)

def smallest_overhead(estimator_name, speed):
    snr_index = LINK_SNR.index(TARGET_SNR)
    passing = [
        name
        for name in REDUCED_CONFIGS
        if (
            (estimator_name, name, speed) in BLER
            and BLER[(estimator_name, name, speed)][snr_index]
            <= TARGET_BLER
        )
    ]
    if not passing:
        return None
    return min(passing, key=lambda name: CFG[name]["eta"])

eta_rows = []
for speed in LINK_SPEEDS:
    for estimator_name in LINK_ESTIMATORS:
        best = smallest_overhead(estimator_name, speed)
        eta_rows.append({
            "speed": speed,
            "estimator": estimator_name,
            "minimum config": best,
            "eta_min [%]": (
                round(100 * CFG[best]["eta"], 2)
                if best is not None else np.nan
            ),
        })
ETA_MIN = pd.DataFrame(eta_rows)
print(ETA_MIN.to_string(index=False))

saving_rows = []
for speed in LINK_SPEEDS:
    cnn_cfg = smallest_overhead("CNN (learned)", speed)
    for reference in ("LS-lin", "LMMSE (site)"):
        ref_cfg = smallest_overhead(reference, speed)
        if cnn_cfg is None or ref_cfg is None:
            absolute = relative = np.nan
        else:
            eta_cnn = CFG[cnn_cfg]["eta"]
            eta_ref = CFG[ref_cfg]["eta"]
            absolute = 100 * (eta_ref - eta_cnn)
            relative = 100 * (eta_ref - eta_cnn) / eta_ref
        saving_rows.append({
            "speed": speed,
            "reference": reference,
            "reference config": ref_cfg,
            "CNN config": cnn_cfg,
            "absolute saving [percentage points]": (
                round(absolute, 2)
            ),
            "relative saving [%]": round(relative, 2),
        })
PILOT_SAVING = pd.DataFrame(saving_rows)
print("\nCNN pilot-overhead saving at 15 dB:")
print(PILOT_SAVING.to_string(index=False))


In [ ]:
# Speed mismatch: target-speed training is the matched control.
MISMATCH_CONFIG = "NR-T1-p0"
MISMATCH_SNR = 15
MISMATCH_STEPS = 800
mismatch_cfg = CFG[MISMATCH_CONFIG]

CNN_SPEED = {}
RIDGE_SPEED = {}
for speed in (3, 120):
    print("Training speed-specific CNN:", speed, "km/h")
    CNN_SPEED[speed] = train_cnn(
        mismatch_cfg,
        steps=MISMATCH_STEPS,
        train_speeds=[speed],
        seed=SEED + speed,
    )
    RIDGE_SPEED[speed] = fit_all_ridge(
        mismatch_cfg, train_speeds=[speed]
    )

def classical_nmse_at(H, cfg, estimator, snr, seed):
    h_hat, _ = estimate_classical(
        H, cfg, estimator, snr, seed=seed
    )
    return nmse_ratio(H, h_hat)

def learned_nmse_at(H, cfg, estimator, snr, seed):
    noise_var = 10 ** (-snr / 10)
    rng = np.random.default_rng(seed)
    noisy = noisy_ls_samples(H, cfg, noise_var, rng)
    h_hat = estimator(noisy, noise_var)
    return nmse_ratio(H, h_hat)

mismatch_rows = []
for source_speed, target_speed in ((3, 120), (120, 3)):
    H = H_test[target_speed]
    same_seed = SEED + source_speed + target_speed

    lmmse_matched = make_classical_estimator(
        mismatch_cfg,
        "LMMSE (site)",
        target_speed,
        covariance_speed=target_speed,
    )
    lmmse_mismatch = make_classical_estimator(
        mismatch_cfg,
        "LMMSE (site)",
        target_speed,
        covariance_speed=source_speed,
    )
    ridge_matched = ridge_estimator(
        RIDGE_SPEED[target_speed],
        mismatch_cfg,
        target_speed,
    )
    ridge_mismatch = ridge_estimator(
        RIDGE_SPEED[source_speed],
        mismatch_cfg,
        source_speed,
    )
    cnn_matched = cnn_estimator(
        CNN_SPEED[target_speed], mismatch_cfg
    )
    cnn_mismatch = cnn_estimator(
        CNN_SPEED[source_speed], mismatch_cfg
    )

    values = {
        "LMMSE matched": classical_nmse_at(
            H, mismatch_cfg, lmmse_matched,
            MISMATCH_SNR, same_seed
        ),
        "LMMSE mismatched": classical_nmse_at(
            H, mismatch_cfg, lmmse_mismatch,
            MISMATCH_SNR, same_seed
        ),
        "Ridge matched": learned_nmse_at(
            H, mismatch_cfg, ridge_matched,
            MISMATCH_SNR, same_seed
        ),
        "Ridge mismatched": learned_nmse_at(
            H, mismatch_cfg, ridge_mismatch,
            MISMATCH_SNR, same_seed
        ),
        "CNN matched": learned_nmse_at(
            H, mismatch_cfg, cnn_matched,
            MISMATCH_SNR, same_seed
        ),
        "CNN mismatched": learned_nmse_at(
            H, mismatch_cfg, cnn_mismatch,
            MISMATCH_SNR, same_seed
        ),
    }

    row = {"train -> test": f"{source_speed} -> {target_speed}"}
    for name, value in values.items():
        row[name + " [dB]"] = round(float(db(value)), 2)
    row["LMMSE degradation [dB]"] = round(
        float(db(values["LMMSE mismatched"])
              - db(values["LMMSE matched"])), 2
    )
    row["Ridge degradation [dB]"] = round(
        float(db(values["Ridge mismatched"])
              - db(values["Ridge matched"])), 2
    )
    row["CNN degradation [dB]"] = round(
        float(db(values["CNN mismatched"])
              - db(values["CNN matched"])), 2
    )
    mismatch_rows.append(row)

SPEED_MISMATCH = pd.DataFrame(mismatch_rows)
print(SPEED_MISMATCH.to_string(index=False))


In [ ]:
# Site B: same UE grid, different rooftop for the mismatch experiment.
BS_B_XY = np.array([345.0, 215.0])
BS_B_MAST_HEIGHT_M = 8.0
bs = scene.get("bs")
saved_bs_position = bs.position.numpy().reshape(-1).copy()
bs_b_z = top_surface_z(*BS_B_XY) + BS_B_MAST_HEIGHT_M
bs_pos_B = np.array([BS_B_XY[0], BS_B_XY[1], bs_b_z])

SITE_B_FILE = "siteB_channels_4x4.pt"
site_b_meta = site_a_meta.copy()
site_b_meta["bs_position"] = bs_pos_B.tolist()

try:
    bs.position = bs_pos_B.tolist()
    bs.look_at(aim_point.tolist())
    los_B = direct_los(scene, outdoor_positions, bs_pos_B)
    used = splits_arr != "buffer"
    print("Site-B LoS fraction:", round(los_B[used].mean(), 3))
    if los_B[used].all() or (~los_B[used]).all():
        raise RuntimeError("Site B needs both LoS and NLoS locations.")

    if os.path.exists(SITE_B_FILE):
        dsB = torch.load(SITE_B_FILE, weights_only=False)
        if not cache_matches(dsB, site_b_meta):
            raise RuntimeError(
                f"{SITE_B_FILE} belongs to another geometry."
            )
        print("Loaded", SITE_B_FILE)
    else:
        dsB = generate_dataset(
            scene,
            outdoor_positions,
            splits_arr,
            loc_dirs,
            los_B,
            site_seed=5000,
        )
        dsB["metadata"] = site_b_meta
        torch.save(dsB, SITE_B_FILE)
        print("Saved", SITE_B_FILE)
finally:
    bs.position = saved_bs_position.tolist()
    bs.look_at(aim_point.tolist())

valid_sets_B = []
for speed in SPEEDS:
    mask = (dsB["speed"] == speed) & dsB["valid_path"]
    valid_sets_B.append(set(dsB["loc"][mask].tolist()))
if not all(s == valid_sets_B[0] for s in valid_sets_B[1:]):
    raise RuntimeError("Site-B valid locations changed between speeds.")

valid_B = (
    (dsB["speed"] == SPEEDS[0])
    & dsB["valid_path"]
)
fraction_B = valid_B.sum() / int((splits_arr != "buffer").sum())
print("Site-B valid-path fraction:", round(float(fraction_B), 3))
if fraction_B < 0.80:
    print("Warning: fewer than 80% of Site-B locations have a path.")
if dsB["los"][valid_B].all() or (~dsB["los"][valid_B]).all():
    raise RuntimeError("Valid Site-B channels need both LoS and NLoS.")

plt.figure(figsize=(7, 6))
plt.scatter(
    outdoor_positions[:, 0], outdoor_positions[:, 1],
    c=np.where(los_B, "tab:blue", "tab:orange"),
    s=18, label="UE grid"
)
plt.scatter(
    bs_pos_A[0], bs_pos_A[1], marker="^", s=130,
    color="black", label="BS A"
)
plt.scatter(
    bs_pos_B[0], bs_pos_B[1], marker="^", s=130,
    color="red", label="BS B"
)
plt.axis("equal")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("BS-location mismatch")
plt.legend()
plt.show()


In [ ]:
# Site-B covariance uses only Site-B training samples.
H_train_B, speed_train_B = view(dsB, M_EVAL, "train")
Rf_B, _ = estimate_covariances(H_train_B)
Rt_B = {
    speed: estimate_covariances(
        H_train_B[speed_train_B == speed]
    )[1]
    for speed in SPEEDS
}
COV_B = {
    "Rf": torch.tensor(Rf_B, device=device),
    "Rt": {
        speed: torch.tensor(Rt_B[speed], device=device)
        for speed in SPEEDS
    },
}

def common_test_channels(dataset_A, dataset_B, M, speed):
    HA, loc_A = view_with_locations(
        dataset_A, M, "test", speed
    )
    HB, loc_B = view_with_locations(
        dataset_B, M, "test", speed
    )
    index_A = {int(loc): i for i, loc in enumerate(loc_A)}
    index_B = {int(loc): i for i, loc in enumerate(loc_B)}
    common = sorted(set(index_A) & set(index_B))
    if not common:
        raise RuntimeError("Sites A and B have no common test locations.")
    HA = np.stack([HA[index_A[loc]] for loc in common])
    HB = np.stack([HB[index_B[loc]] for loc in common])
    return HA, HB, np.asarray(common)

site_rows = []
for speed in (3, 120):
    HA, HB, common_locations = common_test_channels(
        ds, dsB, M_EVAL, speed
    )
    seed = SEED + 7000 + speed

    lmmse_A = make_classical_estimator(
        mismatch_cfg,
        "LMMSE (site)",
        speed,
        covariances=COV_SITE,
    )
    lmmse_B = make_classical_estimator(
        mismatch_cfg,
        "LMMSE (site)",
        speed,
        covariances=COV_B,
    )
    lmmse_tdl = make_classical_estimator(
        mismatch_cfg,
        "LMMSE (TDL-C)",
        speed,
    )
    ridge_A = ridge_estimator(
        FILTERS[MISMATCH_CONFIG],
        mismatch_cfg,
        speed,
    )
    cnn_A = cnn_estimator(
        CNN[MISMATCH_CONFIG], mismatch_cfg
    )

    values = {
        "LMMSE A-cov on A": classical_nmse_at(
            HA, mismatch_cfg, lmmse_A,
            MISMATCH_SNR, seed
        ),
        "LMMSE A-cov on B": classical_nmse_at(
            HB, mismatch_cfg, lmmse_A,
            MISMATCH_SNR, seed
        ),
        "LMMSE B-train cov on B": classical_nmse_at(
            HB, mismatch_cfg, lmmse_B,
            MISMATCH_SNR, seed
        ),
        "TDL-C on B": classical_nmse_at(
            HB, mismatch_cfg, lmmse_tdl,
            MISMATCH_SNR, seed
        ),
        "Ridge A on A": learned_nmse_at(
            HA, mismatch_cfg, ridge_A,
            MISMATCH_SNR, seed
        ),
        "Ridge A on B": learned_nmse_at(
            HB, mismatch_cfg, ridge_A,
            MISMATCH_SNR, seed
        ),
        "CNN A on A": learned_nmse_at(
            HA, mismatch_cfg, cnn_A,
            MISMATCH_SNR, seed
        ),
        "CNN A on B": learned_nmse_at(
            HB, mismatch_cfg, cnn_A,
            MISMATCH_SNR, seed
        ),
    }

    row = {
        "speed": speed,
        "common test locations": len(common_locations),
    }
    for name, value in values.items():
        row[name + " [dB]"] = round(float(db(value)), 2)
    row["Ridge site degradation [dB]"] = round(
        float(db(values["Ridge A on B"])
              - db(values["Ridge A on A"])), 2
    )
    row["CNN site degradation [dB]"] = round(
        float(db(values["CNN A on B"])
              - db(values["CNN A on A"])), 2
    )
    site_rows.append(row)

SITE_MISMATCH = pd.DataFrame(site_rows)
print(SITE_MISMATCH.to_string(index=False))
print(
    "\nA-to-B degradation includes both site difficulty and statistics "
    "mismatch. The Site-B covariance result is the matched LMMSE reference."
)


## Theory extension (EE58009)

Let N=14x288 and approximate the channel degrees of freedom by

D = (B tau_max + 1)(2 f_D T_slot + 1).

With sigma_e^2 = D/(eta N rho), numerical maximization of the assignment's
rate expression can be compared with the high-SNR approximation

eta* approximately sqrt(D/(N ln(rho))).


In [ ]:
N_RE = NSYM * NSC
BANDWIDTH = NSC * SCS

def channel_dof(tau_max, doppler):
    return (
        (BANDWIDTH * tau_max + 1)
        * (2 * doppler * T_SLOT + 1)
    )

def theory_rate(eta, rho, dof):
    error = np.minimum(dof / (eta * N_RE * rho), 1.0)
    return (
        (1 - eta)
        * np.log2(1 + rho * (1 - error) / (1 + rho * error))
    )

def numeric_eta_star(rho, dof):
    lower = max(dof / N_RE * 1.0001, 1e-4)
    grid = np.linspace(lower, 0.6, 4000)
    return grid[np.argmax(theory_rate(grid, rho, dof))]

theory_rows = []
for speed in SPEEDS:
    selected = (
        (ds["split"] == "test")
        & (ds["speed"] == speed)
        & ds["valid_path"]
    )
    delays = ds["tau_max"][selected]
    for label, tau in (
        ("median", np.median(delays)),
        ("90th percentile", np.percentile(delays, 90)),
    ):
        dof = channel_dof(tau, FD_NOM[speed])
        for snr in (10, 20):
            rho = 10 ** (snr / 10)
            theory_rows.append({
                "speed": speed,
                "delay statistic": label,
                "tau_max [ns]": round(tau * 1e9, 1),
                "D": round(dof, 2),
                "SNR": snr,
                "eta* numeric [%]": round(
                    100 * numeric_eta_star(rho, dof), 2
                ),
                "eta* approximation [%]": round(
                    100 * np.sqrt(dof / (N_RE * np.log(rho))), 2
                ),
            })
THEORY = pd.DataFrame(theory_rows)
print(THEORY.to_string(index=False))
print("\nSimulated eta* from Step 8:")
print(ETA_STAR[
    ETA_STAR["estimator"].isin(
        ["LMMSE (site)", "CNN (learned)"]
    )
].to_string(index=False))


## Discussion — adaptive pilot overhead

A practical BS can estimate delay spread from recent DMRS/SRS power-delay
profiles and Doppler from channel autocorrelation. Together with SNR, CQI
and recent BLER it can choose the lowest-overhead pattern that still meets
the time/frequency sampling conditions. If BLER rises at unchanged SNR, it
can increase pilot density.

The learned estimator should only be selected while its residual error
remains close to validation behaviour. The speed- and BS-mismatch tables
quantify this risk. When mismatch is detected, the receiver can return to
the site/TDL LMMSE estimator or use a denser DMRS pattern.

A negative value in the pilot-saving table means that the CNN requires at
least as much pilot overhead as the reference estimator.
